# Modelado segmentado + comparación de modelos — predicción a 3 y 6 meses

## Objetivo

Esta versión mantiene el **diferenciamiento de consumidores** y vuelve a incorporar una comparación objetiva entre:

1. **LightGBM**
2. **XGBoost**
3. **CatBoost**

La comparación se realiza para cada combinación:

`perfil de consumidor × horizonte`

Por tanto, no se obliga a que un único algoritmo sea el mejor para todos los clientes ni para todos los meses futuros.

## Perfiles

Todo el cálculo de los perfiles se realiza dentro del notebook utilizando únicamente información histórica conocida hasta cada fecha de corte:

- `P0_INTERMITENTE`
- `P1_REGULAR`
- `P2_ALTO`
- `P3_GRANDE`
- `P4_INSUFICIENTE`

El cálculo de **grandes consumidores P3** también permanece completamente dentro de este notebook.

## Estrategia por perfil

### P0 — Intermitente
Cada algoritmo utiliza un esquema hurdle:

`clasificador consumo > 0 + regresor de consumo positivo`

### P1 — Regular
Los tres algoritmos utilizan `log1p(consumo)`.

### P2 — Alto
Los tres algoritmos utilizan una formulación Tweedie sobre kWh, para respetar mejor el volumen de consumo.

### P3 — Grande
También utiliza Tweedie, todos los grandes consumidores disponibles y una combinación con baseline estacional.

### P4 — Historia insuficiente
No participa en la competencia de modelos. Utiliza baseline.

## Horizontes

Se construyen modelos independientes para:

- `h1 = t+1`
- `h2 = t+2`
- `h3 = t+3`
- `h4 = t+4`
- `h5 = t+5`
- `h6 = t+6`

## Selección

Para cada `perfil × horizonte`:

1. se entrenan LightGBM, XGBoost y CatBoost con el mismo dataset;
2. se evalúan únicamente en validación temporal;
3. para cada modelo se optimiza `alpha` contra el baseline;
4. se selecciona el menor WAPE de validación;
5. el backtest posterior **no modifica el ganador**.

## Gráficas incluidas

El notebook mantiene y amplía las visualizaciones:

- participación energética por perfil;
- WAPE LightGBM vs XGBoost vs CatBoost por horizonte;
- WAPE por modelo separado para cada perfil;
- WAPE del modelo puro vs versión combinada con baseline;
- consumo real vs predicho por los tres modelos en validación;
- consumo real vs sistema ganador en backtest;
- comparación específica para grandes consumidores;
- dispersión real vs pronosticado cliente a cliente.

## Dependencias

Si alguna librería de modelos no está instalada:

```python
%pip install -U lightgbm xgboost catboost scikit-learn pyarrow joblib
```

Después reinicia el kernel y ejecuta nuevamente desde el inicio.

In [ ]:
# ============================================================
# 1. LIBRERÍAS Y RUTAS
# ============================================================

from pathlib import Path
import gc
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import joblib

try:
    from lightgbm import (
        LGBMRegressor,
        LGBMClassifier,
    )
except ImportError as e:
    raise ImportError(
        "Falta LightGBM. Ejecuta: %pip install -U lightgbm"
    ) from e

try:
    from xgboost import (
        XGBRegressor,
        XGBClassifier,
    )
except ImportError as e:
    raise ImportError(
        "Falta XGBoost. Ejecuta: %pip install -U xgboost"
    ) from e

try:
    from catboost import (
        CatBoostRegressor,
        CatBoostClassifier,
    )
except ImportError as e:
    raise ImportError(
        "Falta CatBoost. Ejecuta: %pip install -U catboost"
    ) from e

warnings.filterwarnings("ignore")

BASE_DIR = Path(
    r"C:\Users\Home\Documents\Datos_Ebsa"
)

PREPROC_DIR = (
    BASE_DIR
    / "03_serie_modelado"
)

RUTA_ENTRADA = (
    PREPROC_DIR
    / "serie_mensual_modelado_preprocesada.parquet"
)

SALIDA_DIR = (
    BASE_DIR
    / "04_pronostico" / "modelo_final"
)

SALIDA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

RUTA_PERFILES_FINAL = (
    SALIDA_DIR
    / "perfiles_consumidores_corte_final.parquet"
)

RUTA_GRANDES_FINAL = (
    SALIDA_DIR
    / "grandes_consumidores_corte_final.parquet"
)

RUTA_AUDITORIA_UMBRALES = (
    SALIDA_DIR
    / "auditoria_umbrales_grandes_consumidores.csv"
)

RUTA_COMPARACION_VALIDACION = (
    SALIDA_DIR
    / "comparacion_modelos_validacion.csv"
)

RUTA_SELECCION_MODELOS = (
    SALIDA_DIR
    / "seleccion_modelo_por_perfil_horizonte.csv"
)

RUTA_TOTALES_VALIDACION = (
    SALIDA_DIR
    / "real_vs_modelos_validacion_totales.csv"
)

RUTA_METRICAS_BACKTEST = (
    SALIDA_DIR
    / "metricas_sistema_ganador_backtest.csv"
)

RUTA_METRICAS_PERFIL = (
    SALIDA_DIR
    / "metricas_sistema_por_perfil_horizonte.csv"
)

RUTA_METRICAS_REGIMEN = (
    SALIDA_DIR
    / "metricas_sistema_por_regimen_horizonte.csv"
)

RUTA_REAL_VS_PRED = (
    SALIDA_DIR
    / "real_vs_pronosticado_sistema_ganador_backtest.parquet"
)

RUTA_MODELOS = (
    SALIDA_DIR
    / "modelos_ganadores_segmentados.joblib"
)

RUTA_PRED_3M = (
    SALIDA_DIR
    / "predicciones_segmentadas_ganadoras_3_meses.parquet"
)

RUTA_PRED_6M = (
    SALIDA_DIR
    / "predicciones_segmentadas_ganadoras_6_meses.parquet"
)

print("Entrada :", RUTA_ENTRADA)
print("Salidas :", SALIDA_DIR)

In [ ]:
# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

SEED = 42

HORIZONTES = [
    1, 2, 3, 4, 5, 6
]

MODELOS_CANDIDATOS = [
    "LightGBM",
    "XGBoost",
    "CatBoost",
]

# ------------------------------------------------------------
# Segmentación
# ------------------------------------------------------------

MIN_MESES_VALIDOS_12 = 6

UMBRAL_MUY_BAJO_KWH = 10.0
UMBRAL_ALTO_KWH = 500.0
UMBRAL_GRANDE_KWH = 5_000.0

PCT_CEROS_INTERMITENTE = 0.50

PERFILES = [
    "P0_INTERMITENTE",
    "P1_REGULAR",
    "P2_ALTO",
    "P3_GRANDE",
    "P4_INSUFICIENTE",
]

PERFILES_MODELADOS = [
    "P0_INTERMITENTE",
    "P1_REGULAR",
    "P2_ALTO",
    "P3_GRANDE",
]

# ------------------------------------------------------------
# Muestreo de entrenamiento
# P2/P3 usan todos los candidatos disponibles.
# ------------------------------------------------------------

MAX_MUESTRA_POR_ORIGEN = {
    "P0_INTERMITENTE": 40_000,
    "P1_REGULAR": 60_000,
    "P2_ALTO": None,
    "P3_GRANDE": None,
}

# ------------------------------------------------------------
# Ventanas temporales
# ------------------------------------------------------------

PRIMER_ORIGEN_TRAIN = pd.Timestamp(
    "2023-01-01"
)

# Los tres modelos se comparan entrenando con targets
# conocidos hasta julio de 2024.
MAX_TARGET_TRAIN_COMPARACION = pd.Timestamp(
    "2024-07-01"
)

# Ventanas utilizadas para seleccionar algoritmo y alpha.
CORTES_VALIDACION = [
    pd.Timestamp("2024-07-01"),
    pd.Timestamp("2025-01-01"),
]

# Backtest completamente posterior.
CORTE_BACKTEST_FINAL = pd.Timestamp(
    "2025-07-01"
)

# Búsqueda de peso ML vs baseline.
GRID_ALPHA = np.round(
    np.linspace(
        0.0,
        1.0,
        21,
    ),
    2,
)

print("MODELOS:", MODELOS_CANDIDATOS)

print("\nPERFILES")
print("-" * 60)

print(
    f"P0: mediana <= {UMBRAL_MUY_BAJO_KWH:,.0f} kWh "
    f"o ceros >= {PCT_CEROS_INTERMITENTE:.0%}"
)

print(
    f"P1: > {UMBRAL_MUY_BAJO_KWH:,.0f} "
    f"y < {UMBRAL_ALTO_KWH:,.0f} kWh"
)

print(
    f"P2: {UMBRAL_ALTO_KWH:,.0f} "
    f"a < {UMBRAL_GRANDE_KWH:,.0f} kWh"
)

print(
    f"P3: >= {UMBRAL_GRANDE_KWH:,.0f} kWh "
    "de mediana histórica 12m"
)

print(
    f"P4: < {MIN_MESES_VALIDOS_12} meses válidos"
)

In [ ]:
# ============================================================
# 3. CARGAR Y VALIDAR ARCHIVO PREPROCESADO
# ============================================================

if not RUTA_ENTRADA.exists():
    raise FileNotFoundError(
        f"No existe el archivo:\n{RUTA_ENTRADA}"
    )

serie = pd.read_parquet(
    RUTA_ENTRADA,
    engine="pyarrow",
)

obligatorias = [
    "NIU",
    "periodo",
    "consumo_kwh_mensual",
]

faltantes = [
    c
    for c in obligatorias
    if c not in serie.columns
]

if faltantes:
    raise ValueError(
        f"Faltan columnas obligatorias: {faltantes}"
    )

serie["NIU"] = (
    serie["NIU"]
    .astype("string")
    .str.strip()
)

serie["periodo"] = pd.to_datetime(
    serie["periodo"],
    errors="coerce",
)

serie["consumo_kwh_mensual"] = pd.to_numeric(
    serie["consumo_kwh_mensual"],
    errors="coerce",
).astype("float32")

duplicados = int(
    serie.duplicated(
        subset=[
            "NIU",
            "periodo",
        ]
    ).sum()
)

negativos = int(
    serie[
        "consumo_kwh_mensual"
    ]
    .lt(0)
    .sum()
)

print("VALIDACIÓN DE ENTRADA")
print("-" * 60)
print(f"Filas                  : {len(serie):,}")
print(f"NIU únicos             : {serie['NIU'].nunique():,}")
print(
    f"Periodo                : "
    f"{serie['periodo'].min():%Y-%m} "
    f"→ {serie['periodo'].max():%Y-%m}"
)
print(f"Duplicados NIU-periodo : {duplicados:,}")
print(
    f"Consumos nulos         : "
    f"{serie['consumo_kwh_mensual'].isna().sum():,}"
)
print(
    f"Consumos cero          : "
    f"{serie['consumo_kwh_mensual'].eq(0).sum():,}"
)
print(f"Consumos negativos     : {negativos:,}")

if duplicados != 0:
    raise ValueError(
        "Existen duplicados NIU-periodo."
    )

if negativos != 0:
    raise ValueError(
        "Hay consumos negativos. Revisar antes de modelar."
    )

In [ ]:
# ============================================================
# 4. CREAR MATRIZ DE CONSUMO NIU x MES
# ============================================================

periodo_min = (
    serie["periodo"]
    .min()
    .to_period("M")
    .to_timestamp()
)

periodo_max = (
    serie["periodo"]
    .max()
    .to_period("M")
    .to_timestamp()
)

meses = pd.date_range(
    start=periodo_min,
    end=periodo_max,
    freq="MS",
)

wide_consumo = (
    serie[
        [
            "NIU",
            "periodo",
            "consumo_kwh_mensual",
        ]
    ]
    .pivot(
        index="NIU",
        columns="periodo",
        values="consumo_kwh_mensual",
    )
    .reindex(
        columns=meses
    )
    .astype("float32")
)

nius = (
    wide_consumo.index
    .astype("string")
    .to_numpy()
)

matriz_consumo = wide_consumo.to_numpy(
    dtype="float32",
    copy=False,
)

mapa_mes = {
    pd.Timestamp(mes): i
    for i, mes in enumerate(
        wide_consumo.columns
    )
}

print("Matriz consumo:", matriz_consumo.shape)
print(
    "Memoria:",
    f"{matriz_consumo.nbytes / 1024**2:,.1f} MB"
)

del wide_consumo
gc.collect()

In [ ]:
# ============================================================
# 5. MATRIZ DE RÉGIMEN OBSERVADO / RECONSTRUIDO
# ============================================================
#
# 1 = reconstruido trimestral
# 0 = observado
# NaN = mes sin fila
#
# Esta variable NO define el perfil de consumo, pero sí entra
# como feature y permite auditar los errores por régimen.
# ============================================================

matriz_reconstruido = None

if (
    "origen_consumo" in serie.columns
    or "consumo_imputado" in serie.columns
):

    if "origen_consumo" in serie.columns:
        flag_origen = (
            serie["origen_consumo"]
            .astype("string")
            .str.contains(
                "reconstru",
                case=False,
                na=False,
            )
        )
    else:
        flag_origen = pd.Series(
            False,
            index=serie.index,
        )

    if "consumo_imputado" in serie.columns:
        flag_imputado = (
            serie["consumo_imputado"]
            .fillna(False)
            .astype(bool)
        )
    else:
        flag_imputado = pd.Series(
            False,
            index=serie.index,
        )

    serie["_es_reconstruido"] = (
        flag_origen
        | flag_imputado
    ).astype("float32")

    wide_regimen = (
        serie[
            [
                "NIU",
                "periodo",
                "_es_reconstruido",
            ]
        ]
        .pivot(
            index="NIU",
            columns="periodo",
            values="_es_reconstruido",
        )
        .reindex(
            index=nius,
            columns=meses,
        )
        .astype("float32")
    )

    matriz_reconstruido = (
        wide_regimen.to_numpy(
            dtype="float32",
            copy=False,
        )
    )

    print(
        "Matriz régimen creada:",
        matriz_reconstruido.shape
    )

    del wide_regimen
    gc.collect()

else:
    print(
        "No existen columnas de procedencia. "
        "El modelado continuará sin feature de régimen."
    )

In [ ]:
# ============================================================
# 6. FUNCIONES TEMPORALES
# ============================================================

def periodo_mes(fecha):
    return (
        pd.Timestamp(fecha)
        .to_period("M")
        .to_timestamp()
    )


def sumar_meses(
    fecha,
    delta,
):
    return (
        periodo_mes(fecha)
        .to_period("M")
        + delta
    ).to_timestamp()


def valores_mes(
    fecha,
    indices=None,
):
    fecha = periodo_mes(fecha)

    if indices is None:
        n = matriz_consumo.shape[0]
    else:
        n = len(indices)

    if fecha not in mapa_mes:
        return np.full(
            n,
            np.nan,
            dtype="float32",
        )

    columna = mapa_mes[fecha]

    if indices is None:
        return matriz_consumo[
            :,
            columna
        ]

    return matriz_consumo[
        indices,
        columna
    ]


def valores_regimen_mes(
    fecha,
    indices,
):
    if matriz_reconstruido is None:
        return np.full(
            len(indices),
            np.nan,
            dtype="float32",
        )

    fecha = periodo_mes(fecha)

    if fecha not in mapa_mes:
        return np.full(
            len(indices),
            np.nan,
            dtype="float32",
        )

    return matriz_reconstruido[
        indices,
        mapa_mes[fecha],
    ]


def target_horizonte(
    fecha_corte,
    horizonte,
    indices=None,
):
    fecha_target = sumar_meses(
        fecha_corte,
        horizonte,
    )

    return (
        valores_mes(
            fecha_target,
            indices,
        ),
        fecha_target,
    )

In [ ]:
# ============================================================
# 7. FUNCIONES DE VENTANA
# ============================================================

def ventana_consumo(
    fecha_corte,
    n_meses,
    indices,
):
    return np.column_stack(
        [
            valores_mes(
                sumar_meses(
                    fecha_corte,
                    -lag,
                ),
                indices,
            )
            for lag in range(n_meses)
        ]
    ).astype("float32")


def ventana_regimen(
    fecha_corte,
    n_meses,
    indices,
):
    return np.column_stack(
        [
            valores_regimen_mes(
                sumar_meses(
                    fecha_corte,
                    -lag,
                ),
                indices,
            )
            for lag in range(n_meses)
        ]
    ).astype("float32")


def media_nan(a):
    cuenta = np.sum(
        ~np.isnan(a),
        axis=1,
    )

    suma = np.nansum(
        a,
        axis=1,
    )

    salida = np.full(
        len(a),
        np.nan,
        dtype="float32",
    )

    mask = cuenta > 0

    salida[mask] = (
        suma[mask]
        / cuenta[mask]
    )

    return salida


def mediana_nan(a):
    salida = np.full(
        len(a),
        np.nan,
        dtype="float32",
    )

    mask = np.any(
        ~np.isnan(a),
        axis=1,
    )

    if mask.any():
        salida[mask] = np.nanmedian(
            a[mask],
            axis=1,
        )

    return salida


def std_nan(a):
    with np.errstate(
        invalid="ignore",
        divide="ignore",
    ):
        return np.nanstd(
            a,
            axis=1,
        ).astype("float32")


def min_nan(a):
    salida = np.full(
        len(a),
        np.nan,
        dtype="float32",
    )

    mask = np.any(
        ~np.isnan(a),
        axis=1,
    )

    if mask.any():
        salida[mask] = np.nanmin(
            a[mask],
            axis=1,
        )

    return salida


def max_nan(a):
    salida = np.full(
        len(a),
        np.nan,
        dtype="float32",
    )

    mask = np.any(
        ~np.isnan(a),
        axis=1,
    )

    if mask.any():
        salida[mask] = np.nanmax(
            a[mask],
            axis=1,
        )

    return salida

# Cálculo de perfiles de consumidor

La siguiente función es la pieza central del notebook.

Para cada NIU y fecha de corte calcula, usando únicamente información histórica:

- meses válidos;
- media 12m;
- mediana 12m;
- máximo 12m;
- desviación 12m;
- porcentaje de ceros;
- porcentaje de meses reconstruidos;
- perfil de consumidor.

La categoría `P3_GRANDE` se obtiene aquí mismo a partir de:

`mediana_12m >= UMBRAL_GRANDE_KWH`

Por defecto `UMBRAL_GRANDE_KWH = 5.000`.

La mediana evita clasificar como gran consumidor a un cliente que solo tuvo un pico aislado.

In [ ]:
# ============================================================
# 8. CALCULAR PERFIL DEL CONSUMIDOR EN UNA FECHA DE CORTE
# ============================================================

def calcular_perfiles(
    fecha_corte,
    indices=None,
):
    if indices is None:
        indices = np.arange(
            matriz_consumo.shape[0]
        )

    v12 = ventana_consumo(
        fecha_corte,
        12,
        indices,
    )

    validos12 = np.sum(
        ~np.isnan(v12),
        axis=1,
    )

    media12 = media_nan(v12)
    mediana12 = mediana_nan(v12)
    max12 = max_nan(v12)
    std12 = std_nan(v12)

    ceros12 = np.sum(
        np.where(
            np.isnan(v12),
            False,
            v12 == 0,
        ),
        axis=1,
    )

    pct_ceros12 = np.full(
        len(indices),
        np.nan,
        dtype="float32",
    )

    mask_validos = validos12 > 0

    pct_ceros12[mask_validos] = (
        ceros12[mask_validos]
        / validos12[mask_validos]
    )

    if matriz_reconstruido is not None:
        vr12 = ventana_regimen(
            fecha_corte,
            12,
            indices,
        )

        pct_reconstruido12 = media_nan(
            vr12
        )
    else:
        pct_reconstruido12 = np.full(
            len(indices),
            np.nan,
            dtype="float32",
        )

    perfiles = np.full(
        len(indices),
        "P4_INSUFICIENTE",
        dtype=object,
    )

    historia_ok = (
        validos12
        >= MIN_MESES_VALIDOS_12
    )

    intermitente = (
        historia_ok
        & (
            (mediana12 <= UMBRAL_MUY_BAJO_KWH)
            | (
                pct_ceros12
                >= PCT_CEROS_INTERMITENTE
            )
        )
    )

    regular = (
        historia_ok
        & ~intermitente
        & (
            mediana12
            < UMBRAL_ALTO_KWH
        )
    )

    alto = (
        historia_ok
        & ~intermitente
        & (
            mediana12
            >= UMBRAL_ALTO_KWH
        )
        & (
            mediana12
            < UMBRAL_GRANDE_KWH
        )
    )

    grande = (
        historia_ok
        & (
            mediana12
            >= UMBRAL_GRANDE_KWH
        )
    )

    perfiles[
        intermitente
    ] = "P0_INTERMITENTE"

    perfiles[
        regular
    ] = "P1_REGULAR"

    perfiles[
        alto
    ] = "P2_ALTO"

    perfiles[
        grande
    ] = "P3_GRANDE"

    return pd.DataFrame(
        {
            "indice":
                indices,

            "NIU":
                nius[
                    indices
                ],

            "perfil":
                perfiles,

            "meses_validos_12m":
                validos12.astype(
                    "int8"
                ),

            "media_12m_kwh":
                media12,

            "mediana_12m_kwh":
                mediana12,

            "max_12m_kwh":
                max12,

            "std_12m_kwh":
                std12,

            "pct_ceros_12m":
                pct_ceros12,

            "pct_reconstruido_12m":
                pct_reconstruido12,
        }
    )

In [ ]:
# ============================================================
# 9. AUDITORÍA DE PERFILES EN EL ÚLTIMO MES DISPONIBLE
# ============================================================

FECHA_CORTE_FINAL = periodo_mes(
    periodo_max
)

perfiles_final = calcular_perfiles(
    FECHA_CORTE_FINAL
)

actual_final = valores_mes(
    FECHA_CORTE_FINAL
)

perfiles_final[
    "consumo_actual_kwh"
] = actual_final[
    perfiles_final["indice"]
]

resumen_perfiles_final = (
    perfiles_final
    .groupby(
        "perfil",
        as_index=False,
    )
    .agg(
        NIU=(
            "NIU",
            "nunique"
        ),
        consumo_actual_total_kwh=(
            "consumo_actual_kwh",
            "sum"
        ),
        mediana_historica_promedio_kwh=(
            "mediana_12m_kwh",
            "mean"
        ),
        media_historica_promedio_kwh=(
            "media_12m_kwh",
            "mean"
        ),
    )
)

total_energia = (
    resumen_perfiles_final[
        "consumo_actual_total_kwh"
    ].sum()
)

resumen_perfiles_final[
    "participacion_energia_pct"
] = np.where(
    total_energia > 0,
    (
        resumen_perfiles_final[
            "consumo_actual_total_kwh"
        ]
        / total_energia
        * 100
    ),
    np.nan,
)

display(
    resumen_perfiles_final
)

perfiles_final.to_parquet(
    RUTA_PERFILES_FINAL,
    index=False,
    engine="pyarrow",
)

In [ ]:
# ============================================================
# 10. CÁLCULO Y AUDITORÍA DE GRANDES CONSUMIDORES
# ============================================================
#
# Todo el cálculo de grandes consumidores queda dentro
# del notebook.
# ============================================================

grandes_final = (
    perfiles_final[
        perfiles_final["perfil"]
        .eq("P3_GRANDE")
    ]
    .copy()
    .sort_values(
        "media_12m_kwh",
        ascending=False,
    )
)

energia_grandes = (
    grandes_final[
        "consumo_actual_kwh"
    ].sum()
)

energia_total = (
    perfiles_final[
        "consumo_actual_kwh"
    ].sum()
)

participacion_grandes = (
    energia_grandes
    / energia_total
    * 100
    if energia_total > 0
    else np.nan
)

print("GRANDES CONSUMIDORES")
print("-" * 60)
print(
    f"Umbral mediana 12m : "
    f"{UMBRAL_GRANDE_KWH:,.0f} kWh"
)
print(
    f"NIU P3             : "
    f"{len(grandes_final):,}"
)
print(
    f"Energía último mes : "
    f"{energia_grandes:,.0f} kWh"
)
print(
    f"% energía total    : "
    f"{participacion_grandes:.2f}%"
)

print("\nDistribución histórica de P3:")

display(
    grandes_final[
        [
            "media_12m_kwh",
            "mediana_12m_kwh",
            "max_12m_kwh",
            "std_12m_kwh",
            "pct_reconstruido_12m",
        ]
    ]
    .describe(
        percentiles=[
            .25,
            .50,
            .75,
            .90,
            .95,
            .99,
        ]
    )
    .T
)

print("\nTop 20 grandes consumidores:")

display(
    grandes_final[
        [
            "NIU",
            "consumo_actual_kwh",
            "media_12m_kwh",
            "mediana_12m_kwh",
            "max_12m_kwh",
            "pct_reconstruido_12m",
        ]
    ]
    .head(20)
)

grandes_final.to_parquet(
    RUTA_GRANDES_FINAL,
    index=False,
    engine="pyarrow",
)

In [ ]:
# ============================================================
# 11. SENSIBILIDAD DEL UMBRAL DE GRAN CONSUMIDOR
# ============================================================
#
# Esta tabla NO cambia automáticamente el umbral.
# Sirve para ver cómo cambia el tamaño y la participación
# energética de P3 con diferentes cortes.
# ============================================================

umbrales_grande = [
    2_000,
    3_000,
    5_000,
    7_500,
    10_000,
]

filas_umbrales = []

for umbral in umbrales_grande:

    mask = (
        perfiles_final[
            "meses_validos_12m"
        ]
        .ge(
            MIN_MESES_VALIDOS_12
        )
        & perfiles_final[
            "mediana_12m_kwh"
        ]
        .ge(
            umbral
        )
    )

    temp = perfiles_final[
        mask
    ]

    energia = (
        temp[
            "consumo_actual_kwh"
        ].sum()
    )

    filas_umbrales.append(
        {
            "umbral_mediana_12m_kwh":
                umbral,

            "NIU":
                temp[
                    "NIU"
                ].nunique(),

            "energia_actual_kwh":
                energia,

            "participacion_energia_pct":
                (
                    energia
                    / energia_total
                    * 100
                    if energia_total > 0
                    else np.nan
                ),
        }
    )

auditoria_umbrales = pd.DataFrame(
    filas_umbrales
)

display(
    auditoria_umbrales
)

auditoria_umbrales.to_csv(
    RUTA_AUDITORIA_UMBRALES,
    index=False,
    encoding="utf-8-sig",
)

In [ ]:
# ============================================================
# 12. GRÁFICA DE COMPOSICIÓN POR PERFIL
# ============================================================

plot_perfiles = (
    resumen_perfiles_final
    .sort_values(
        "participacion_energia_pct",
        ascending=False,
    )
)

ax = plot_perfiles.plot(
    x="perfil",
    y="participacion_energia_pct",
    kind="bar",
    figsize=(10, 5),
    legend=False,
)

ax.set_title(
    "Participación del consumo actual por perfil"
)

ax.set_xlabel(
    "Perfil"
)

ax.set_ylabel(
    "Participación de energía (%)"
)

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Features de los modelos especializados

Cada modelo recibe únicamente información conocida hasta la fecha de corte:

- consumo actual;
- lags 1–12 y lag 24;
- mismo mes del año anterior;
- mismo mes de hace 2 años;
- medias, medianas y volatilidad;
- ceros recientes;
- crecimiento reciente;
- régimen observado/reconstruido;
- mes objetivo.

`NIU` nunca se utiliza como predictor.

In [ ]:
# ============================================================
# 13. CONSTRUIR FEATURES
# ============================================================

FEATURES = [
    "consumo_actual",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_4",
    "lag_5",
    "lag_6",
    "lag_12",
    "lag_24",
    "media_3m",
    "media_6m",
    "media_12m",
    "mediana_12m",
    "std_3m",
    "std_6m",
    "std_12m",
    "min_6m",
    "max_6m",
    "max_12m",
    "pct_ceros_6m",
    "pct_ceros_12m",
    "variacion_1m",
    "variacion_3m",
    "ratio_actual_media6",
    "mismo_mes_anio_anterior",
    "mismo_mes_2_anios",
    "reconstruido_actual",
    "pct_reconstruido_12m",
    "mes_objetivo",
    "mes_sin",
    "mes_cos",
]


def crear_features(
    fecha_corte,
    horizonte,
    indices,
):
    fecha_corte = periodo_mes(
        fecha_corte
    )

    fecha_objetivo = sumar_meses(
        fecha_corte,
        horizonte,
    )

    actual = valores_mes(
        fecha_corte,
        indices,
    )

    lags = {
        lag: valores_mes(
            sumar_meses(
                fecha_corte,
                -lag,
            ),
            indices,
        )
        for lag in [
            1, 2, 3, 4, 5, 6, 12, 24
        ]
    }

    v3 = ventana_consumo(
        fecha_corte,
        3,
        indices,
    )

    v6 = ventana_consumo(
        fecha_corte,
        6,
        indices,
    )

    v12 = ventana_consumo(
        fecha_corte,
        12,
        indices,
    )

    media6 = media_nan(v6)

    validos6 = np.sum(
        ~np.isnan(v6),
        axis=1,
    )

    validos12 = np.sum(
        ~np.isnan(v12),
        axis=1,
    )

    ceros6 = np.sum(
        np.where(
            np.isnan(v6),
            False,
            v6 == 0,
        ),
        axis=1,
    )

    ceros12 = np.sum(
        np.where(
            np.isnan(v12),
            False,
            v12 == 0,
        ),
        axis=1,
    )

    pct_ceros6 = np.full(
        len(indices),
        np.nan,
        dtype="float32",
    )

    pct_ceros12 = np.full(
        len(indices),
        np.nan,
        dtype="float32",
    )

    mask6 = validos6 > 0
    mask12 = validos12 > 0

    pct_ceros6[mask6] = (
        ceros6[mask6]
        / validos6[mask6]
    )

    pct_ceros12[mask12] = (
        ceros12[mask12]
        / validos12[mask12]
    )

    ratio = np.full(
        len(indices),
        np.nan,
        dtype="float32",
    )

    mask_ratio = (
        np.isfinite(actual)
        & np.isfinite(media6)
        & (media6 != 0)
    )

    ratio[mask_ratio] = (
        actual[mask_ratio]
        / media6[mask_ratio]
    )

    mismo_mes_1a = valores_mes(
        sumar_meses(
            fecha_objetivo,
            -12,
        ),
        indices,
    )

    mismo_mes_2a = valores_mes(
        sumar_meses(
            fecha_objetivo,
            -24,
        ),
        indices,
    )

    reconstruido_actual = (
        valores_regimen_mes(
            fecha_corte,
            indices,
        )
    )

    if matriz_reconstruido is not None:
        vr12 = ventana_regimen(
            fecha_corte,
            12,
            indices,
        )

        pct_reconstruido12 = (
            media_nan(vr12)
        )
    else:
        pct_reconstruido12 = np.full(
            len(indices),
            np.nan,
            dtype="float32",
        )

    mes_obj = fecha_objetivo.month

    X = pd.DataFrame(
        {
            "consumo_actual":
                actual,

            "lag_1":
                lags[1],

            "lag_2":
                lags[2],

            "lag_3":
                lags[3],

            "lag_4":
                lags[4],

            "lag_5":
                lags[5],

            "lag_6":
                lags[6],

            "lag_12":
                lags[12],

            "lag_24":
                lags[24],

            "media_3m":
                media_nan(v3),

            "media_6m":
                media6,

            "media_12m":
                media_nan(v12),

            "mediana_12m":
                mediana_nan(v12),

            "std_3m":
                std_nan(v3),

            "std_6m":
                std_nan(v6),

            "std_12m":
                std_nan(v12),

            "min_6m":
                min_nan(v6),

            "max_6m":
                max_nan(v6),

            "max_12m":
                max_nan(v12),

            "pct_ceros_6m":
                pct_ceros6,

            "pct_ceros_12m":
                pct_ceros12,

            "variacion_1m":
                (
                    actual
                    - lags[1]
                ).astype("float32"),

            "variacion_3m":
                (
                    actual
                    - lags[3]
                ).astype("float32"),

            "ratio_actual_media6":
                ratio,

            "mismo_mes_anio_anterior":
                mismo_mes_1a,

            "mismo_mes_2_anios":
                mismo_mes_2a,

            "reconstruido_actual":
                reconstruido_actual,

            "pct_reconstruido_12m":
                pct_reconstruido12,

            "mes_objetivo":
                np.full(
                    len(indices),
                    mes_obj,
                    dtype="float32",
                ),

            "mes_sin":
                np.full(
                    len(indices),
                    np.sin(
                        2
                        * np.pi
                        * mes_obj
                        / 12
                    ),
                    dtype="float32",
                ),

            "mes_cos":
                np.full(
                    len(indices),
                    np.cos(
                        2
                        * np.pi
                        * mes_obj
                        / 12
                    ),
                    dtype="float32",
                ),
        }
    )

    return X[
        FEATURES
    ], fecha_objetivo

In [ ]:
# ============================================================
# 14. MÉTRICAS Y BASELINE
# ============================================================

def metricas_regresion(
    real,
    pred,
):
    real = np.asarray(
        real,
        dtype="float64",
    )

    pred = np.asarray(
        pred,
        dtype="float64",
    )

    mask = (
        np.isfinite(real)
        & np.isfinite(pred)
    )

    real = real[mask]
    pred = pred[mask]

    if len(real) == 0:
        return {
            "n": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "WAPE_pct": np.nan,
            "sMAPE_pct": np.nan,
            "R2": np.nan,
            "sesgo_pct": np.nan,
        }

    mae = mean_absolute_error(
        real,
        pred,
    )

    rmse = np.sqrt(
        mean_squared_error(
            real,
            pred,
        )
    )

    suma_real = np.abs(
        real
    ).sum()

    wape = (
        np.abs(
            real - pred
        ).sum()
        / suma_real
        * 100
        if suma_real > 0
        else np.nan
    )

    denom = (
        np.abs(real)
        + np.abs(pred)
    )

    mask_smape = denom > 0

    smape = (
        np.mean(
            2
            * np.abs(
                real[mask_smape]
                - pred[mask_smape]
            )
            / denom[mask_smape]
        )
        * 100
        if mask_smape.any()
        else 0.0
    )

    r2 = (
        r2_score(
            real,
            pred,
        )
        if len(real) > 1
        else np.nan
    )

    sesgo = (
        (
            pred.sum()
            - real.sum()
        )
        / real.sum()
        * 100
        if real.sum() != 0
        else np.nan
    )

    return {
        "n": int(len(real)),
        "MAE": float(mae),
        "RMSE": float(rmse),
        "WAPE_pct": float(wape),
        "sMAPE_pct": float(smape),
        "R2": float(r2),
        "sesgo_pct": float(sesgo),
    }


def baseline_hibrido(
    fecha_corte,
    horizonte,
    indices,
):
    fecha_target = sumar_meses(
        fecha_corte,
        horizonte,
    )

    estacional = valores_mes(
        sumar_meses(
            fecha_target,
            -12,
        ),
        indices,
    )

    actual = valores_mes(
        fecha_corte,
        indices,
    )

    return np.where(
        np.isfinite(estacional),
        estacional,
        actual,
    ).astype("float32")

# Comparación de LightGBM, XGBoost y CatBoost

A partir de este punto los tres algoritmos reciben exactamente el mismo `X_train` y `y_train` para cada combinación de perfil y horizonte.

La competencia se realiza **solo en validación temporal**.

Para cada modelo se calculan dos resultados:

1. **ML puro**
2. **ML + baseline**, optimizando `alpha`

La selección final utiliza el menor `WAPE_blend_validacion_pct`. En caso de empate se usa como desempate el WAPE del ML puro.

In [ ]:
# ============================================================
# 15. FÁBRICAS DE MODELOS POR ALGORITMO
# ============================================================

def nuevo_clasificador(
    algoritmo,
):
    if algoritmo == "LightGBM":
        return LGBMClassifier(
            objective="binary",
            n_estimators=350,
            learning_rate=0.05,
            num_leaves=63,
            min_child_samples=100,
            subsample=0.90,
            colsample_bytree=0.90,
            reg_lambda=0.50,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        )

    if algoritmo == "XGBoost":
        return XGBClassifier(
            objective="binary:logistic",
            n_estimators=350,
            learning_rate=0.05,
            max_depth=8,
            min_child_weight=20,
            subsample=0.90,
            colsample_bytree=0.90,
            reg_lambda=1.0,
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
            verbosity=0,
        )

    if algoritmo == "CatBoost":
        return CatBoostClassifier(
            loss_function="Logloss",
            iterations=350,
            learning_rate=0.05,
            depth=8,
            l2_leaf_reg=5.0,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )

    raise ValueError(
        f"Algoritmo no reconocido: {algoritmo}"
    )


def nuevo_regresor_log(
    algoritmo,
):
    if algoritmo == "LightGBM":
        return LGBMRegressor(
            objective="regression",
            n_estimators=500,
            learning_rate=0.04,
            num_leaves=63,
            min_child_samples=100,
            subsample=0.90,
            colsample_bytree=0.90,
            reg_alpha=0.05,
            reg_lambda=0.50,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        )

    if algoritmo == "XGBoost":
        return XGBRegressor(
            objective="reg:squarederror",
            n_estimators=500,
            learning_rate=0.04,
            max_depth=8,
            min_child_weight=20,
            subsample=0.90,
            colsample_bytree=0.90,
            reg_alpha=0.05,
            reg_lambda=1.0,
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
            verbosity=0,
        )

    if algoritmo == "CatBoost":
        return CatBoostRegressor(
            loss_function="RMSE",
            iterations=500,
            learning_rate=0.04,
            depth=8,
            l2_leaf_reg=5.0,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )

    raise ValueError(
        f"Algoritmo no reconocido: {algoritmo}"
    )


def nuevo_regresor_tweedie(
    algoritmo,
    gran_consumidor=False,
):
    iteraciones = (
        650
        if gran_consumidor
        else 550
    )

    if algoritmo == "LightGBM":
        return LGBMRegressor(
            objective="tweedie",
            tweedie_variance_power=1.5,
            n_estimators=iteraciones,
            learning_rate=0.035,
            num_leaves=(
                31
                if gran_consumidor
                else 63
            ),
            min_child_samples=(
                20
                if gran_consumidor
                else 50
            ),
            subsample=0.95,
            colsample_bytree=0.95,
            reg_alpha=0.05,
            reg_lambda=1.0,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1,
        )

    if algoritmo == "XGBoost":
        return XGBRegressor(
            objective="reg:tweedie",
            tweedie_variance_power=1.5,
            n_estimators=iteraciones,
            learning_rate=0.035,
            max_depth=(
                6
                if gran_consumidor
                else 8
            ),
            min_child_weight=(
                10
                if gran_consumidor
                else 20
            ),
            subsample=0.95,
            colsample_bytree=0.95,
            reg_alpha=0.05,
            reg_lambda=1.0,
            tree_method="hist",
            random_state=SEED,
            n_jobs=-1,
            verbosity=0,
        )

    if algoritmo == "CatBoost":
        return CatBoostRegressor(
            loss_function="Tweedie:variance_power=1.5",
            iterations=iteraciones,
            learning_rate=0.035,
            depth=(
                7
                if gran_consumidor
                else 8
            ),
            l2_leaf_reg=5.0,
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )

    raise ValueError(
        f"Algoritmo no reconocido: {algoritmo}"
    )

In [ ]:
# ============================================================
# 16. CONSTRUIR TRAIN PARA UN PERFIL Y HORIZONTE
# ============================================================

def construir_train_perfil_h(
    perfil,
    horizonte,
    max_target_train,
    seed,
):
    rng = np.random.default_rng(
        seed
    )

    max_target_train = periodo_mes(
        max_target_train
    )

    ultimo_origen = sumar_meses(
        max_target_train,
        -horizonte,
    )

    origenes = pd.date_range(
        start=PRIMER_ORIGEN_TRAIN,
        end=ultimo_origen,
        freq="MS",
    )

    X_partes = []
    y_partes = []
    auditoria = []

    limite = (
        MAX_MUESTRA_POR_ORIGEN[
            perfil
        ]
    )

    for numero, origen in enumerate(
        origenes,
        start=1,
    ):
        perfiles_origen = calcular_perfiles(
            origen
        )

        idx_perfil = (
            perfiles_origen.loc[
                perfiles_origen[
                    "perfil"
                ].eq(perfil),
                "indice",
            ]
            .to_numpy(
                dtype="int64"
            )
        )

        if len(idx_perfil) == 0:
            continue

        y_all, fecha_target = (
            target_horizonte(
                origen,
                horizonte,
                idx_perfil,
            )
        )

        candidatos = idx_perfil[
            np.isfinite(
                y_all
            )
        ]

        if len(candidatos) == 0:
            continue

        if (
            limite is not None
            and len(candidatos) > limite
        ):
            seleccion = rng.choice(
                candidatos,
                size=limite,
                replace=False,
            )
        else:
            seleccion = candidatos

        y_sel, _ = target_horizonte(
            origen,
            horizonte,
            seleccion,
        )

        X_sel, _ = crear_features(
            origen,
            horizonte,
            seleccion,
        )

        X_partes.append(
            X_sel
        )

        y_partes.append(
            y_sel.astype(
                "float32"
            )
        )

        auditoria.append(
            {
                "perfil":
                    perfil,

                "horizonte":
                    horizonte,

                "origen":
                    origen,

                "target":
                    fecha_target,

                "candidatos":
                    len(candidatos),

                "muestra":
                    len(seleccion),
            }
        )

        if (
            numero == 1
            or numero % 6 == 0
            or numero == len(origenes)
        ):
            print(
                f"{perfil} | h={horizonte} | "
                f"{numero}/{len(origenes)} | "
                f"origen={origen:%Y-%m} | "
                f"muestra={len(seleccion):,}"
            )

        del perfiles_origen
        del X_sel
        gc.collect()

    if not X_partes:
        return (
            pd.DataFrame(
                columns=FEATURES
            ),
            np.array(
                [],
                dtype="float32",
            ),
            pd.DataFrame(),
        )

    X = pd.concat(
        X_partes,
        ignore_index=True,
    )

    y = np.concatenate(
        y_partes
    ).astype("float32")

    auditoria = pd.DataFrame(
        auditoria
    )

    del X_partes
    del y_partes
    gc.collect()

    return X, y, auditoria

In [ ]:
# ============================================================
# 17. ENTRENAR / PREDECIR UN ALGORITMO SEGÚN EL PERFIL
# ============================================================

def entrenar_modelo_algoritmo(
    perfil,
    algoritmo,
    X,
    y,
):
    if len(y) == 0:
        return None

    # --------------------------------------------------------
    # P0: hurdle = clasificador + regresor positivo
    # --------------------------------------------------------

    if perfil == "P0_INTERMITENTE":

        y_bin = (
            y > 0
        ).astype("int8")

        bundle = {
            "perfil":
                perfil,

            "algoritmo":
                algoritmo,

            "tipo":
                "hurdle",

            "prob_constante":
                float(
                    y_bin.mean()
                ),

            "classifier":
                None,

            "regressor":
                None,
        }

        if np.unique(
            y_bin
        ).size >= 2:

            clf = nuevo_clasificador(
                algoritmo
            )

            clf.fit(
                X,
                y_bin,
            )

            bundle[
                "classifier"
            ] = clf

        positivos = (
            y > 0
        )

        if positivos.any():

            reg = nuevo_regresor_log(
                algoritmo
            )

            reg.fit(
                X.loc[
                    positivos
                ],
                np.log1p(
                    y[
                        positivos
                    ]
                ),
            )

            bundle[
                "regressor"
            ] = reg

        return bundle

    # --------------------------------------------------------
    # P1: log1p
    # --------------------------------------------------------

    if perfil == "P1_REGULAR":

        reg = nuevo_regresor_log(
            algoritmo
        )

        reg.fit(
            X,
            np.log1p(
                np.maximum(
                    y,
                    0,
                )
            ),
        )

        return {
            "perfil":
                perfil,

            "algoritmo":
                algoritmo,

            "tipo":
                "log",

            "regressor":
                reg,
        }

    # --------------------------------------------------------
    # P2/P3: Tweedie en kWh
    # --------------------------------------------------------

    if perfil in [
        "P2_ALTO",
        "P3_GRANDE",
    ]:

        reg = nuevo_regresor_tweedie(
            algoritmo=algoritmo,
            gran_consumidor=(
                perfil
                == "P3_GRANDE"
            ),
        )

        reg.fit(
            X,
            np.maximum(
                y,
                0,
            ),
        )

        return {
            "perfil":
                perfil,

            "algoritmo":
                algoritmo,

            "tipo":
                "tweedie",

            "regressor":
                reg,
        }

    raise ValueError(
        f"Perfil no reconocido: {perfil}"
    )


def predecir_bundle(
    bundle,
    X,
):
    if bundle is None:
        return np.full(
            len(X),
            np.nan,
            dtype="float32",
        )

    tipo = bundle[
        "tipo"
    ]

    algoritmo = bundle[
        "algoritmo"
    ]

    if tipo == "hurdle":

        if (
            bundle[
                "classifier"
            ]
            is None
        ):
            p_pos = np.full(
                len(X),
                bundle[
                    "prob_constante"
                ],
                dtype="float32",
            )
        else:
            p_pos = (
                bundle[
                    "classifier"
                ]
                .predict_proba(
                    X
                )[:, 1]
                .astype(
                    "float32"
                )
            )

        if (
            bundle[
                "regressor"
            ]
            is None
        ):
            consumo_positivo = np.zeros(
                len(X),
                dtype="float32",
            )
        else:
            consumo_positivo = np.maximum(
                np.expm1(
                    bundle[
                        "regressor"
                    ].predict(
                        X
                    )
                ),
                0,
            ).astype(
                "float32"
            )

        return (
            p_pos
            * consumo_positivo
        ).astype(
            "float32"
        )

    if tipo == "log":

        return np.maximum(
            np.expm1(
                bundle[
                    "regressor"
                ].predict(
                    X
                )
            ),
            0,
        ).astype(
            "float32"
        )

    if tipo == "tweedie":

        if algoritmo == "CatBoost":

            pred = (
                bundle[
                    "regressor"
                ]
                .predict(
                    X,
                    prediction_type="Exponent",
                )
            )

        else:

            pred = (
                bundle[
                    "regressor"
                ]
                .predict(
                    X
                )
            )

        return np.maximum(
            pred,
            0,
        ).astype(
            "float32"
        )

    raise ValueError(
        f"Tipo de bundle no reconocido: {tipo}"
    )

In [ ]:
# ============================================================
# 18. COMPONENTES DE MÉTRICAS ACUMULABLES
# ============================================================

def componentes_vacios():
    return {
        "n":
            0,

        "abs_error_sum":
            0.0,

        "sq_error_sum":
            0.0,

        "abs_real_sum":
            0.0,

        "real_sum":
            0.0,

        "real_sq_sum":
            0.0,

        "pred_sum":
            0.0,

        "smape_sum":
            0.0,

        "smape_n":
            0,
    }


def actualizar_componentes(
    comp,
    real,
    pred,
):
    real = np.asarray(
        real,
        dtype="float64",
    )

    pred = np.asarray(
        pred,
        dtype="float64",
    )

    mask = (
        np.isfinite(real)
        & np.isfinite(pred)
    )

    real = real[mask]
    pred = pred[mask]

    if len(real) == 0:
        return comp

    error = (
        pred
        - real
    )

    comp[
        "n"
    ] += len(
        real
    )

    comp[
        "abs_error_sum"
    ] += float(
        np.abs(
            error
        ).sum()
    )

    comp[
        "sq_error_sum"
    ] += float(
        np.square(
            error
        ).sum()
    )

    comp[
        "abs_real_sum"
    ] += float(
        np.abs(
            real
        ).sum()
    )

    comp[
        "real_sum"
    ] += float(
        real.sum()
    )

    comp[
        "real_sq_sum"
    ] += float(
        np.square(
            real
        ).sum()
    )

    comp[
        "pred_sum"
    ] += float(
        pred.sum()
    )

    denom = (
        np.abs(
            real
        )
        + np.abs(
            pred
        )
    )

    mask_smape = (
        denom > 0
    )

    if mask_smape.any():

        comp[
            "smape_sum"
        ] += float(
            (
                2
                * np.abs(
                    real[
                        mask_smape
                    ]
                    - pred[
                        mask_smape
                    ]
                )
                / denom[
                    mask_smape
                ]
            ).sum()
        )

        comp[
            "smape_n"
        ] += int(
            mask_smape.sum()
        )

    return comp


def metricas_componentes(
    comp,
):
    n = comp[
        "n"
    ]

    if n == 0:
        return {
            "n": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "WAPE_pct": np.nan,
            "sMAPE_pct": np.nan,
            "R2": np.nan,
            "sesgo_pct": np.nan,
        }

    mae = (
        comp[
            "abs_error_sum"
        ]
        / n
    )

    rmse = np.sqrt(
        comp[
            "sq_error_sum"
        ]
        / n
    )

    wape = (
        comp[
            "abs_error_sum"
        ]
        / comp[
            "abs_real_sum"
        ]
        * 100
        if comp[
            "abs_real_sum"
        ] > 0
        else np.nan
    )

    smape = (
        comp[
            "smape_sum"
        ]
        / comp[
            "smape_n"
        ]
        * 100
        if comp[
            "smape_n"
        ] > 0
        else np.nan
    )

    media_real = (
        comp[
            "real_sum"
        ]
        / n
    )

    sst = (
        comp[
            "real_sq_sum"
        ]
        - n
        * media_real**2
    )

    r2 = (
        1
        - (
            comp[
                "sq_error_sum"
            ]
            / sst
        )
        if sst > 0
        else np.nan
    )

    sesgo = (
        (
            comp[
                "pred_sum"
            ]
            - comp[
                "real_sum"
            ]
        )
        / comp[
            "real_sum"
        ]
        * 100
        if comp[
            "real_sum"
        ] != 0
        else np.nan
    )

    return {
        "n":
            int(
                n
            ),

        "MAE":
            float(
                mae
            ),

        "RMSE":
            float(
                rmse
            ),

        "WAPE_pct":
            float(
                wape
            ),

        "sMAPE_pct":
            float(
                smape
            ),

        "R2":
            float(
                r2
            ),

        "sesgo_pct":
            float(
                sesgo
            ),
    }

In [ ]:
# ============================================================
# 19. OBTENER DATASET DE VALIDACIÓN PARA UN GRUPO
# ============================================================

def obtener_validacion_grupo(
    perfil,
    horizonte,
    fecha_corte,
):
    perfiles_corte = calcular_perfiles(
        fecha_corte
    )

    indices = (
        perfiles_corte.loc[
            perfiles_corte[
                "perfil"
            ].eq(
                perfil
            ),
            "indice",
        ]
        .to_numpy(
            dtype="int64"
        )
    )

    if len(indices) == 0:
        return None

    y_real, fecha_target = (
        target_horizonte(
            fecha_corte,
            horizonte,
            indices,
        )
    )

    mask = np.isfinite(
        y_real
    )

    indices = indices[
        mask
    ]

    y_real = y_real[
        mask
    ].astype(
        "float32"
    )

    if len(indices) == 0:
        return None

    X, _ = crear_features(
        fecha_corte,
        horizonte,
        indices,
    )

    baseline = baseline_hibrido(
        fecha_corte,
        horizonte,
        indices,
    )

    return {
        "indices":
            indices,

        "X":
            X,

        "real":
            y_real,

        "baseline":
            baseline,

        "fecha_target":
            fecha_target,
    }

# Entrenamiento y selección de los tres algoritmos

Esta es la parte central de la comparación.

Para cada `perfil × horizonte`:

- se construye el train una sola vez;
- se entrenan los tres modelos;
- se evalúan en los dos cortes de validación;
- se calcula WAPE puro;
- se prueba `alpha = 0.00 ... 1.00`;
- se obtiene el mejor blend de cada algoritmo;
- se elige el algoritmo ganador.

Los modelos perdedores se liberan de memoria después de evaluarlos. Esto evita conservar decenas de modelos simultáneamente.

In [ ]:
# ============================================================
# 20. COMPARAR MODELOS Y SELECCIONAR GANADOR
# ============================================================

filas_comparacion = []

filas_seleccion = []

filas_totales_validacion = []

# Métricas globales para las gráficas por algoritmo/horizonte.
acum_global_ml = {
    (
        algoritmo,
        h,
    ):
        componentes_vacios()
    for algoritmo in MODELOS_CANDIDATOS
    for h in HORIZONTES
}

acum_global_blend = {
    (
        algoritmo,
        h,
    ):
        componentes_vacios()
    for algoritmo in MODELOS_CANDIDATOS
    for h in HORIZONTES
}

acum_global_baseline = {
    h:
        componentes_vacios()
    for h in HORIZONTES
}


for perfil in PERFILES_MODELADOS:

    for h in HORIZONTES:

        print("\n" + "=" * 80)

        print(
            f"COMPARACIÓN | {perfil} | h={h}"
        )

        print("=" * 80)

        inicio_grupo = time.time()

        # ----------------------------------------------------
        # TRAIN COMÚN PARA LOS TRES ALGORITMOS
        # ----------------------------------------------------

        X_train, y_train, audit = (
            construir_train_perfil_h(
                perfil=perfil,
                horizonte=h,
                max_target_train=
                    MAX_TARGET_TRAIN_COMPARACION,
                seed=(
                    SEED
                    + h
                ),
            )
        )

        print(
            "Train:",
            X_train.shape,
            y_train.shape
        )

        if len(y_train) == 0:
            print(
                "Sin datos suficientes. "
                "Se omite este grupo."
            )

            del X_train
            del y_train
            gc.collect()

            continue

        resultados_algoritmos = []

        # ----------------------------------------------------
        # CADA ALGORITMO COMPITE CON EL MISMO TRAIN
        # ----------------------------------------------------

        for algoritmo in MODELOS_CANDIDATOS:

            print(
                f"\n→ Entrenando {algoritmo}"
            )

            inicio_modelo = time.time()

            bundle = entrenar_modelo_algoritmo(
                perfil=perfil,
                algoritmo=algoritmo,
                X=X_train,
                y=y_train,
            )

            comp_ml = componentes_vacios()

            comp_baseline = componentes_vacios()

            comp_alpha = {
                float(
                    alpha
                ):
                    componentes_vacios()
                for alpha
                in GRID_ALPHA
            }

            # -----------------------------------------------
            # Validación en cortes completamente temporales
            # -----------------------------------------------

            for corte in CORTES_VALIDACION:

                val = obtener_validacion_grupo(
                    perfil=perfil,
                    horizonte=h,
                    fecha_corte=corte,
                )

                if val is None:
                    continue

                pred_ml = predecir_bundle(
                    bundle,
                    val[
                        "X"
                    ],
                )

                real = val[
                    "real"
                ]

                baseline = val[
                    "baseline"
                ]

                actualizar_componentes(
                    comp_ml,
                    real,
                    pred_ml,
                )

                actualizar_componentes(
                    comp_baseline,
                    real,
                    baseline,
                )

                actualizar_componentes(
                    acum_global_ml[
                        (
                            algoritmo,
                            h,
                        )
                    ],
                    real,
                    pred_ml,
                )

                actualizar_componentes(
                    acum_global_baseline[
                        h
                    ],
                    real,
                    baseline,
                )

                for alpha in GRID_ALPHA:

                    alpha = float(
                        alpha
                    )

                    pred_blend = (
                        alpha
                        * pred_ml
                        + (
                            1
                            - alpha
                        )
                        * baseline
                    )

                    actualizar_componentes(
                        comp_alpha[
                            alpha
                        ],
                        real,
                        pred_blend,
                    )

                filas_totales_validacion.append(
                    {
                        "perfil":
                            perfil,

                        "horizonte":
                            h,

                        "modelo":
                            algoritmo,

                        "fecha_corte":
                            periodo_mes(
                                corte
                            ),

                        "fecha_target":
                            val[
                                "fecha_target"
                            ],

                        "n":
                            len(
                                real
                            ),

                        "real_total_kwh":
                            float(
                                np.sum(
                                    real
                                )
                            ),

                        "pred_ml_total_kwh":
                            float(
                                np.sum(
                                    pred_ml
                                )
                            ),

                        "baseline_total_kwh":
                            float(
                                np.sum(
                                    baseline
                                )
                            ),
                    }
                )

                del val
                del pred_ml
                gc.collect()

            # -----------------------------------------------
            # Métricas puras
            # -----------------------------------------------

            met_ml = metricas_componentes(
                comp_ml
            )

            met_baseline = metricas_componentes(
                comp_baseline
            )

            # -----------------------------------------------
            # Optimizar alpha de ESTE algoritmo
            # -----------------------------------------------

            candidatos_alpha = []

            for alpha, comp in comp_alpha.items():

                met_alpha = metricas_componentes(
                    comp
                )

                candidatos_alpha.append(
                    (
                        met_alpha[
                            "WAPE_pct"
                        ],
                        alpha,
                        comp,
                    )
                )

            candidatos_alpha = [
                x
                for x in candidatos_alpha
                if np.isfinite(
                    x[0]
                )
            ]

            if candidatos_alpha:

                candidatos_alpha.sort(
                    key=lambda x: (
                        x[0],
                        -x[1],
                    )
                )

                mejor_wape_blend, mejor_alpha, mejor_comp = (
                    candidatos_alpha[
                        0
                    ]
                )

                met_blend = metricas_componentes(
                    mejor_comp
                )

            else:

                mejor_alpha = 1.0
                met_blend = met_ml

            # Global del algoritmo usando el alpha óptimo
            # de este perfil/horizonte.

            for corte in CORTES_VALIDACION:

                val = obtener_validacion_grupo(
                    perfil=perfil,
                    horizonte=h,
                    fecha_corte=corte,
                )

                if val is None:
                    continue

                pred_ml = predecir_bundle(
                    bundle,
                    val[
                        "X"
                    ],
                )

                pred_blend = (
                    mejor_alpha
                    * pred_ml
                    + (
                        1
                        - mejor_alpha
                    )
                    * val[
                        "baseline"
                    ]
                )

                actualizar_componentes(
                    acum_global_blend[
                        (
                            algoritmo,
                            h,
                        )
                    ],
                    val[
                        "real"
                    ],
                    pred_blend,
                )

                del val
                del pred_ml
                del pred_blend
                gc.collect()

            fila = {
                "perfil":
                    perfil,

                "horizonte":
                    h,

                "modelo":
                    algoritmo,

                "n_validacion":
                    met_ml[
                        "n"
                    ],

                "MAE_ML":
                    met_ml[
                        "MAE"
                    ],

                "RMSE_ML":
                    met_ml[
                        "RMSE"
                    ],

                "WAPE_ML_pct":
                    met_ml[
                        "WAPE_pct"
                    ],

                "sMAPE_ML_pct":
                    met_ml[
                        "sMAPE_pct"
                    ],

                "R2_ML":
                    met_ml[
                        "R2"
                    ],

                "sesgo_ML_pct":
                    met_ml[
                        "sesgo_pct"
                    ],

                "WAPE_baseline_pct":
                    met_baseline[
                        "WAPE_pct"
                    ],

                "alpha_ml_optimo":
                    mejor_alpha,

                "alpha_baseline_optimo":
                    1
                    - mejor_alpha,

                "WAPE_blend_validacion_pct":
                    met_blend[
                        "WAPE_pct"
                    ],

                "MAE_blend_validacion":
                    met_blend[
                        "MAE"
                    ],

                "sesgo_blend_validacion_pct":
                    met_blend[
                        "sesgo_pct"
                    ],

                "tiempo_modelo_min":
                    (
                        time.time()
                        - inicio_modelo
                    )
                    / 60,
            }

            filas_comparacion.append(
                fila
            )

            resultados_algoritmos.append(
                fila
            )

            print(
                f"{algoritmo:<10} | "
                f"WAPE ML={met_ml['WAPE_pct']:.2f}% | "
                f"alpha={mejor_alpha:.2f} | "
                f"WAPE blend={met_blend['WAPE_pct']:.2f}%"
            )

            del bundle
            gc.collect()

        # ----------------------------------------------------
        # GANADOR DEL PERFIL/HORIZONTE
        # ----------------------------------------------------

        tabla_grupo = pd.DataFrame(
            resultados_algoritmos
        )

        if not tabla_grupo.empty:

            tabla_grupo = (
                tabla_grupo
                .sort_values(
                    [
                        "WAPE_blend_validacion_pct",
                        "WAPE_ML_pct",
                    ],
                    ascending=True,
                )
                .reset_index(
                    drop=True
                )
            )

            ganador = (
                tabla_grupo
                .iloc[0]
            )

            filas_seleccion.append(
                {
                    "perfil":
                        perfil,

                    "horizonte":
                        h,

                    "modelo_ganador":
                        ganador[
                            "modelo"
                        ],

                    "alpha_ml":
                        ganador[
                            "alpha_ml_optimo"
                        ],

                    "alpha_baseline":
                        ganador[
                            "alpha_baseline_optimo"
                        ],

                    "WAPE_ML_validacion_pct":
                        ganador[
                            "WAPE_ML_pct"
                        ],

                    "WAPE_blend_validacion_pct":
                        ganador[
                            "WAPE_blend_validacion_pct"
                        ],

                    "WAPE_baseline_validacion_pct":
                        ganador[
                            "WAPE_baseline_pct"
                        ],
                }
            )

            print(
                "\nGANADOR:",
                ganador[
                    "modelo"
                ],
                "| alpha ML =",
                f"{ganador['alpha_ml_optimo']:.2f}",
                "| WAPE blend =",
                f"{ganador['WAPE_blend_validacion_pct']:.2f}%"
            )

        print(
            "Tiempo grupo:",
            f"{(time.time() - inicio_grupo) / 60:.2f} min"
        )

        del X_train
        del y_train
        gc.collect()

In [ ]:
# ============================================================
# 21. TABLAS DE COMPARACIÓN Y SELECCIÓN
# ============================================================

comparacion_validacion = pd.DataFrame(
    filas_comparacion
)

seleccion_modelos = pd.DataFrame(
    filas_seleccion
)

totales_validacion = pd.DataFrame(
    filas_totales_validacion
)

comparacion_validacion[
    "ganador"
] = False

for fila in seleccion_modelos.itertuples(
    index=False
):

    mask = (
        comparacion_validacion[
            "perfil"
        ].eq(
            fila.perfil
        )
        & comparacion_validacion[
            "horizonte"
        ].eq(
            fila.horizonte
        )
        & comparacion_validacion[
            "modelo"
        ].eq(
            fila.modelo_ganador
        )
    )

    comparacion_validacion.loc[
        mask,
        "ganador",
    ] = True

print("COMPARACIÓN COMPLETA")

display(
    comparacion_validacion
    .sort_values(
        [
            "perfil",
            "horizonte",
            "WAPE_blend_validacion_pct",
        ]
    )
)

print("\nMODELO GANADOR POR PERFIL/HORIZONTE")

display(
    seleccion_modelos
    .sort_values(
        [
            "perfil",
            "horizonte",
        ]
    )
)

comparacion_validacion.to_csv(
    RUTA_COMPARACION_VALIDACION,
    index=False,
    encoding="utf-8-sig",
)

seleccion_modelos.to_csv(
    RUTA_SELECCION_MODELOS,
    index=False,
    encoding="utf-8-sig",
)

totales_validacion.to_csv(
    RUTA_TOTALES_VALIDACION,
    index=False,
    encoding="utf-8-sig",
)

In [ ]:
# ============================================================
# 22. RESUMEN GLOBAL DE VALIDACIÓN POR MODELO/HORIZONTE
# ============================================================

filas_global_validacion = []

for h in HORIZONTES:

    met_base = metricas_componentes(
        acum_global_baseline[
            h
        ]
    )

    for algoritmo in MODELOS_CANDIDATOS:

        met_ml = metricas_componentes(
            acum_global_ml[
                (
                    algoritmo,
                    h,
                )
            ]
        )

        met_blend = metricas_componentes(
            acum_global_blend[
                (
                    algoritmo,
                    h,
                )
            ]
        )

        filas_global_validacion.append(
            {
                "horizonte":
                    h,

                "modelo":
                    algoritmo,

                "WAPE_ML_pct":
                    met_ml[
                        "WAPE_pct"
                    ],

                "WAPE_blend_pct":
                    met_blend[
                        "WAPE_pct"
                    ],

                "WAPE_baseline_pct":
                    met_base[
                        "WAPE_pct"
                    ],

                "MAE_ML":
                    met_ml[
                        "MAE"
                    ],

                "R2_ML":
                    met_ml[
                        "R2"
                    ],

                "sesgo_ML_pct":
                    met_ml[
                        "sesgo_pct"
                    ],
            }
        )

global_validacion = pd.DataFrame(
    filas_global_validacion
)

display(
    global_validacion
)

In [ ]:
# ============================================================
# 23. GRÁFICA: WAPE ML PURO POR MODELO Y HORIZONTE
# ============================================================

tabla_plot = (
    global_validacion
    .pivot(
        index="horizonte",
        columns="modelo",
        values="WAPE_ML_pct",
    )
)

# Baseline una sola vez por horizonte.
baseline_plot = (
    global_validacion
    .groupby(
        "horizonte"
    )[
        "WAPE_baseline_pct"
    ]
    .first()
)

ax = tabla_plot.plot(
    marker="o",
    figsize=(11, 5),
)

ax.plot(
    baseline_plot.index,
    baseline_plot.values,
    marker="o",
    linestyle="--",
    label="Baseline",
)

ax.set_title(
    "Validación: WAPE de LightGBM vs XGBoost vs CatBoost"
)

ax.set_xlabel(
    "Horizonte (meses)"
)

ax.set_ylabel(
    "WAPE (%)"
)

ax.legend()
ax.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 24. GRÁFICA: WAPE DESPUÉS DE OPTIMIZAR ALPHA
# ============================================================

tabla_plot_blend = (
    global_validacion
    .pivot(
        index="horizonte",
        columns="modelo",
        values="WAPE_blend_pct",
    )
)

ax = tabla_plot_blend.plot(
    marker="o",
    figsize=(11, 5),
)

ax.plot(
    baseline_plot.index,
    baseline_plot.values,
    marker="o",
    linestyle="--",
    label="Baseline",
)

ax.set_title(
    "Validación: WAPE de cada modelo después del blend"
)

ax.set_xlabel(
    "Horizonte (meses)"
)

ax.set_ylabel(
    "WAPE (%)"
)

ax.legend()
ax.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 25. GRÁFICAS POR PERFIL: COMPARACIÓN DE MODELOS
# ============================================================
#
# Se crea una figura independiente por perfil.
# ============================================================

for perfil in PERFILES_MODELADOS:

    temp = (
        comparacion_validacion[
            comparacion_validacion[
                "perfil"
            ].eq(
                perfil
            )
        ]
        .pivot(
            index="horizonte",
            columns="modelo",
            values="WAPE_blend_validacion_pct",
        )
    )

    base_perfil = (
        comparacion_validacion[
            comparacion_validacion[
                "perfil"
            ].eq(
                perfil
            )
        ]
        .groupby(
            "horizonte"
        )[
            "WAPE_baseline_pct"
        ]
        .first()
    )

    ax = temp.plot(
        marker="o",
        figsize=(11, 5),
    )

    ax.plot(
        base_perfil.index,
        base_perfil.values,
        marker="o",
        linestyle="--",
        label="Baseline",
    )

    ax.set_title(
        f"{perfil}: comparación de modelos por horizonte"
    )

    ax.set_xlabel(
        "Horizonte (meses)"
    )

    ax.set_ylabel(
        "WAPE validación (%)"
    )

    ax.legend()
    ax.grid(
        alpha=0.25
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 26. GRÁFICA: REAL VS PRONOSTICADO POR MODELO EN VALIDACIÓN
# ============================================================

totales_modelos = (
    totales_validacion
    .groupby(
        [
            "modelo",
            "fecha_target",
        ],
        as_index=False,
    )
    .agg(
        real_total_kwh=(
            "real_total_kwh",
            "sum"
        ),
        pred_total_kwh=(
            "pred_ml_total_kwh",
            "sum"
        ),
        baseline_total_kwh=(
            "baseline_total_kwh",
            "sum"
        ),
    )
)

real_validacion = (
    totales_modelos
    .groupby(
        "fecha_target",
        as_index=False,
    )[
        "real_total_kwh"
    ]
    .first()
    .sort_values(
        "fecha_target"
    )
)

plt.figure(
    figsize=(13, 6)
)

plt.plot(
    real_validacion[
        "fecha_target"
    ],
    real_validacion[
        "real_total_kwh"
    ],
    marker="o",
    linewidth=2,
    label="Real",
)

for algoritmo in MODELOS_CANDIDATOS:

    temp = (
        totales_modelos[
            totales_modelos[
                "modelo"
            ].eq(
                algoritmo
            )
        ]
        .sort_values(
            "fecha_target"
        )
    )

    plt.plot(
        temp[
            "fecha_target"
        ],
        temp[
            "pred_total_kwh"
        ],
        marker="o",
        label=algoritmo,
    )

baseline_validacion = (
    totales_modelos
    .groupby(
        "fecha_target",
        as_index=False,
    )[
        "baseline_total_kwh"
    ]
    .first()
    .sort_values(
        "fecha_target"
    )
)

plt.plot(
    baseline_validacion[
        "fecha_target"
    ],
    baseline_validacion[
        "baseline_total_kwh"
    ],
    marker="o",
    linestyle="--",
    label="Baseline",
)

plt.title(
    "Validación temporal: consumo real vs pronóstico de cada modelo"
)

plt.xlabel(
    "Mes objetivo"
)

plt.ylabel(
    "Consumo total (kWh)"
)

plt.legend()
plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 27. GRÁFICA: GRANDES CONSUMIDORES - MODELOS VS REAL
# ============================================================

p3_totales = (
    totales_validacion[
        totales_validacion[
            "perfil"
        ].eq(
            "P3_GRANDE"
        )
    ]
    .groupby(
        [
            "modelo",
            "fecha_target",
        ],
        as_index=False,
    )
    .agg(
        real_total_kwh=(
            "real_total_kwh",
            "sum"
        ),
        pred_total_kwh=(
            "pred_ml_total_kwh",
            "sum"
        ),
        baseline_total_kwh=(
            "baseline_total_kwh",
            "sum"
        ),
    )
)

if not p3_totales.empty:

    real_p3 = (
        p3_totales
        .groupby(
            "fecha_target",
            as_index=False,
        )[
            "real_total_kwh"
        ]
        .first()
        .sort_values(
            "fecha_target"
        )
    )

    plt.figure(
        figsize=(13, 6)
    )

    plt.plot(
        real_p3[
            "fecha_target"
        ],
        real_p3[
            "real_total_kwh"
        ],
        marker="o",
        linewidth=2,
        label="Real P3",
    )

    for algoritmo in MODELOS_CANDIDATOS:

        temp = (
            p3_totales[
                p3_totales[
                    "modelo"
                ].eq(
                    algoritmo
                )
            ]
            .sort_values(
                "fecha_target"
            )
        )

        plt.plot(
            temp[
                "fecha_target"
            ],
            temp[
                "pred_total_kwh"
            ],
            marker="o",
            label=algoritmo,
        )

    base_p3 = (
        p3_totales
        .groupby(
            "fecha_target",
            as_index=False,
        )[
            "baseline_total_kwh"
        ]
        .first()
        .sort_values(
            "fecha_target"
        )
    )

    plt.plot(
        base_p3[
            "fecha_target"
        ],
        base_p3[
            "baseline_total_kwh"
        ],
        marker="o",
        linestyle="--",
        label="Baseline P3",
    )

    plt.title(
        "P3 grandes consumidores: real vs modelos en validación"
    )

    plt.xlabel(
        "Mes objetivo"
    )

    plt.ylabel(
        "Consumo total P3 (kWh)"
    )

    plt.legend()

    plt.xticks(
        rotation=45
    )

    plt.tight_layout()
    plt.show()

## Gráficas adicionales — comparación de errores por segmento (validación)

Estas gráficas complementan las líneas de WAPE por horizonte ya calculadas: comparan,
perfil por perfil, el modelo finalmente seleccionado contra el baseline, y muestran de
un vistazo qué algoritmo ganó en cada combinación `perfil × horizonte`.


In [ ]:
# ============================================================
# 25b. GRAFICA: WAPE DEL GANADOR VS BASELINE POR PERFIL (BARRAS)
# ============================================================

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 9),
    sharey=False,
)

axes = axes.flatten()

ancho = 0.35

for ax, perfil in zip(axes, PERFILES_MODELADOS):

    temp = (
        seleccion_modelos[
            seleccion_modelos["perfil"].eq(perfil)
        ]
        .sort_values("horizonte")
    )

    x = np.arange(len(temp))

    ax.bar(
        x - ancho / 2,
        temp["WAPE_blend_validacion_pct"],
        width=ancho,
        label="Sistema (ganador)",
        color="#2E7D32",
    )

    ax.bar(
        x + ancho / 2,
        temp["WAPE_baseline_validacion_pct"],
        width=ancho,
        label="Baseline",
        color="#9E9E9E",
    )

    for xi, (modelo, wape) in enumerate(
        zip(temp["modelo_ganador"], temp["WAPE_blend_validacion_pct"])
    ):
        ax.text(
            xi - ancho / 2,
            wape + 0.5,
            modelo,
            ha="center",
            va="bottom",
            fontsize=8,
            rotation=90,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(temp["horizonte"])
    ax.set_title(perfil)
    ax.set_xlabel("Horizonte (meses)")
    ax.set_ylabel("WAPE validación (%)")
    ax.grid(alpha=0.25, axis="y")
    ax.legend(fontsize=8)

fig.suptitle(
    "Validación: WAPE del sistema ganador vs baseline, por perfil y horizonte",
    fontsize=13,
)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 25c. MAPA DE SELECCION: QUE ALGORITMO GANO (VALIDACION)
# ============================================================

pivot_wape = (
    seleccion_modelos
    .pivot(index="perfil", columns="horizonte", values="WAPE_blend_validacion_pct")
    .reindex(PERFILES_MODELADOS)
)

pivot_modelo = (
    seleccion_modelos
    .pivot(index="perfil", columns="horizonte", values="modelo_ganador")
    .reindex(PERFILES_MODELADOS)
)

fig, ax = plt.subplots(figsize=(10, 5))

im = ax.imshow(
    pivot_wape.to_numpy(dtype="float64"),
    cmap="RdYlGn_r",
    aspect="auto",
)

ax.set_xticks(range(len(pivot_wape.columns)))
ax.set_xticklabels(pivot_wape.columns)
ax.set_yticks(range(len(pivot_wape.index)))
ax.set_yticklabels(pivot_wape.index)
ax.set_xlabel("Horizonte (meses)")
ax.set_title("Validación: algoritmo ganador y WAPE del sistema (%)")

for i in range(pivot_wape.shape[0]):
    for j in range(pivot_wape.shape[1]):
        wape = pivot_wape.iat[i, j]
        modelo = pivot_modelo.iat[i, j]
        if pd.notna(wape):
            ax.text(
                j, i,
                f"{modelo}\n{wape:.1f}%",
                ha="center", va="center",
                fontsize=8, color="black",
            )

fig.colorbar(im, ax=ax, label="WAPE (%)")
plt.tight_layout()
plt.show()


# Backtest del sistema ganador

Los algoritmos ya fueron elegidos utilizando únicamente validación.

Ahora se reentrena **solo el ganador** de cada `perfil × horizonte` utilizando toda la información disponible antes del backtest, es decir, targets conocidos hasta julio de 2025.

Después se predice agosto de 2025 a enero de 2026 y se compara contra la realidad.

El backtest no cambia ni el algoritmo ganador ni el alpha.

In [ ]:
# ============================================================
# 28. MAPAS DE SELECCIÓN
# ============================================================

mapa_seleccion = {
    (
        fila.perfil,
        int(
            fila.horizonte
        ),
    ):
        {
            "modelo":
                fila.modelo_ganador,

            "alpha_ml":
                float(
                    fila.alpha_ml
                ),

            "alpha_baseline":
                float(
                    fila.alpha_baseline
                ),
        }
    for fila
    in seleccion_modelos.itertuples(
        index=False
    )
}

display(
    seleccion_modelos
)

In [ ]:
# ============================================================
# 29. REENTRENAR GANADORES PARA BACKTEST
# ============================================================

modelos_backtest = {}

for perfil in PERFILES_MODELADOS:

    for h in HORIZONTES:

        config = mapa_seleccion[
            (
                perfil,
                h,
            )
        ]

        algoritmo = config[
            "modelo"
        ]

        print("\n" + "=" * 75)

        print(
            f"BACKTEST TRAIN | {perfil} | h={h} | {algoritmo}"
        )

        print("=" * 75)

        inicio = time.time()

        X_train, y_train, _ = (
            construir_train_perfil_h(
                perfil=perfil,
                horizonte=h,
                max_target_train=
                    CORTE_BACKTEST_FINAL,
                seed=(
                    SEED
                    + 100
                    + h
                ),
            )
        )

        bundle = entrenar_modelo_algoritmo(
            perfil=perfil,
            algoritmo=algoritmo,
            X=X_train,
            y=y_train,
        )

        modelos_backtest[
            (
                perfil,
                h,
            )
        ] = bundle

        print(
            "Train:",
            X_train.shape,
            "| tiempo:",
            f"{(time.time() - inicio) / 60:.2f} min"
        )

        del X_train
        del y_train
        gc.collect()

In [ ]:
# ============================================================
# 30. GENERAR BACKTEST DEL SISTEMA GANADOR
# ============================================================

perfiles_backtest = calcular_perfiles(
    CORTE_BACKTEST_FINAL
)

partes_backtest = []

for h in HORIZONTES:

    fecha_target = sumar_meses(
        CORTE_BACKTEST_FINAL,
        h,
    )

    for perfil in PERFILES:

        indices = (
            perfiles_backtest.loc[
                perfiles_backtest[
                    "perfil"
                ].eq(
                    perfil
                ),
                "indice",
            ]
            .to_numpy(
                dtype="int64"
            )
        )

        if len(indices) == 0:
            continue

        y_real, _ = target_horizonte(
            CORTE_BACKTEST_FINAL,
            h,
            indices,
        )

        mask = np.isfinite(
            y_real
        )

        indices = indices[
            mask
        ]

        y_real = y_real[
            mask
        ].astype(
            "float32"
        )

        if len(indices) == 0:
            continue

        baseline = baseline_hibrido(
            CORTE_BACKTEST_FINAL,
            h,
            indices,
        )

        if perfil == "P4_INSUFICIENTE":

            algoritmo = "Baseline"

            alpha_ml = 0.0

            pred_ml = baseline.copy()

            pred_final = baseline.copy()

        else:

            config = mapa_seleccion[
                (
                    perfil,
                    h,
                )
            ]

            algoritmo = config[
                "modelo"
            ]

            alpha_ml = config[
                "alpha_ml"
            ]

            X, _ = crear_features(
                CORTE_BACKTEST_FINAL,
                h,
                indices,
            )

            pred_ml = predecir_bundle(
                modelos_backtest[
                    (
                        perfil,
                        h,
                    )
                ],
                X,
            )

            pred_final = (
                alpha_ml
                * pred_ml
                + (
                    1
                    - alpha_ml
                )
                * baseline
            )

            pred_final = np.maximum(
                pred_final,
                0,
            ).astype(
                "float32"
            )

            del X

        regimen_actual = valores_regimen_mes(
            CORTE_BACKTEST_FINAL,
            indices,
        )

        regimen = np.where(
            regimen_actual == 1,
            "reconstruido",
            np.where(
                regimen_actual == 0,
                "observado",
                "sin_dato",
            ),
        )

        temp = pd.DataFrame(
            {
                "NIU":
                    nius[
                        indices
                    ],

                "fecha_corte":
                    CORTE_BACKTEST_FINAL,

                "horizonte":
                    h,

                "fecha_target":
                    fecha_target,

                "perfil":
                    perfil,

                "regimen_actual":
                    regimen,

                "modelo_ganador":
                    algoritmo,

                "alpha_ml":
                    alpha_ml,

                "real_kwh":
                    y_real,

                "pred_ml_kwh":
                    pred_ml,

                "baseline_kwh":
                    baseline,

                "pred_final_kwh":
                    pred_final,
            }
        )

        partes_backtest.append(
            temp
        )

        del temp
        gc.collect()

backtest = pd.concat(
    partes_backtest,
    ignore_index=True,
)

print(
    "Filas backtest:",
    f"{len(backtest):,}"
)

display(
    backtest.head(20)
)

In [ ]:
# ============================================================
# 31. MÉTRICAS BACKTEST POR PERFIL/HORIZONTE
# ============================================================

filas_metricas_perfil = []

for (
    perfil,
    h,
), g in backtest.groupby(
    [
        "perfil",
        "horizonte",
    ]
):

    met_final = metricas_regresion(
        g[
            "real_kwh"
        ],
        g[
            "pred_final_kwh"
        ],
    )

    met_ml = metricas_regresion(
        g[
            "real_kwh"
        ],
        g[
            "pred_ml_kwh"
        ],
    )

    met_base = metricas_regresion(
        g[
            "real_kwh"
        ],
        g[
            "baseline_kwh"
        ],
    )

    modelo_ganador = (
        g[
            "modelo_ganador"
        ]
        .mode()
        .iloc[0]
    )

    alpha = (
        g[
            "alpha_ml"
        ]
        .iloc[0]
    )

    filas_metricas_perfil.append(
        {
            "perfil":
                perfil,

            "horizonte":
                h,

            "modelo_ganador":
                modelo_ganador,

            "alpha_ml":
                alpha,

            "n":
                met_final[
                    "n"
                ],

            "WAPE_final_pct":
                met_final[
                    "WAPE_pct"
                ],

            "WAPE_ML_pct":
                met_ml[
                    "WAPE_pct"
                ],

            "WAPE_baseline_pct":
                met_base[
                    "WAPE_pct"
                ],

            "MAE_final":
                met_final[
                    "MAE"
                ],

            "RMSE_final":
                met_final[
                    "RMSE"
                ],

            "R2_final":
                met_final[
                    "R2"
                ],

            "sesgo_final_pct":
                met_final[
                    "sesgo_pct"
                ],
        }
    )

metricas_perfil_backtest = pd.DataFrame(
    filas_metricas_perfil
)

display(
    metricas_perfil_backtest
    .sort_values(
        [
            "horizonte",
            "perfil",
        ]
    )
)

metricas_perfil_backtest.to_csv(
    RUTA_METRICAS_PERFIL,
    index=False,
    encoding="utf-8-sig",
)

In [ ]:
# ============================================================
# 32. MÉTRICAS GLOBALES DEL SISTEMA GANADOR
# ============================================================

filas_global_backtest = []

for h, g in backtest.groupby(
    "horizonte"
):

    final = metricas_regresion(
        g[
            "real_kwh"
        ],
        g[
            "pred_final_kwh"
        ],
    )

    ml = metricas_regresion(
        g[
            "real_kwh"
        ],
        g[
            "pred_ml_kwh"
        ],
    )

    baseline = metricas_regresion(
        g[
            "real_kwh"
        ],
        g[
            "baseline_kwh"
        ],
    )

    filas_global_backtest.append(
        {
            "horizonte":
                h,

            "n":
                final[
                    "n"
                ],

            "WAPE_final_pct":
                final[
                    "WAPE_pct"
                ],

            "WAPE_ML_ganadores_pct":
                ml[
                    "WAPE_pct"
                ],

            "WAPE_baseline_pct":
                baseline[
                    "WAPE_pct"
                ],

            "MAE_final":
                final[
                    "MAE"
                ],

            "RMSE_final":
                final[
                    "RMSE"
                ],

            "R2_final":
                final[
                    "R2"
                ],

            "sesgo_final_pct":
                final[
                    "sesgo_pct"
                ],
        }
    )

metricas_globales_backtest = pd.DataFrame(
    filas_global_backtest
)

display(
    metricas_globales_backtest
)

metricas_globales_backtest.to_csv(
    RUTA_METRICAS_BACKTEST,
    index=False,
    encoding="utf-8-sig",
)

In [ ]:
# ============================================================
# 33. MÉTRICAS POR RÉGIMEN OBSERVADO / RECONSTRUIDO
# ============================================================

filas_regimen = []

for (
    regimen,
    h,
), g in backtest.groupby(
    [
        "regimen_actual",
        "horizonte",
    ]
):

    met = metricas_regresion(
        g[
            "real_kwh"
        ],
        g[
            "pred_final_kwh"
        ],
    )

    filas_regimen.append(
        {
            "regimen":
                regimen,

            "horizonte":
                h,

            **met,
        }
    )

metricas_regimen = pd.DataFrame(
    filas_regimen
)

display(
    metricas_regimen
    .sort_values(
        [
            "regimen",
            "horizonte",
        ]
    )
)

metricas_regimen.to_csv(
    RUTA_METRICAS_REGIMEN,
    index=False,
    encoding="utf-8-sig",
)

In [ ]:
# ============================================================
# 34. REAL VS PRONOSTICADO CLIENTE A CLIENTE
# ============================================================

backtest[
    "error_kwh"
] = (
    backtest[
        "pred_final_kwh"
    ]
    - backtest[
        "real_kwh"
    ]
)

backtest[
    "error_abs_kwh"
] = np.abs(
    backtest[
        "error_kwh"
    ]
)

backtest[
    "error_pct"
] = np.where(
    backtest[
        "real_kwh"
    ] > 0,
    (
        backtest[
            "error_kwh"
        ]
        / backtest[
            "real_kwh"
        ]
        * 100
    ),
    np.nan,
)

real_vs_pred = backtest[
    [
        "NIU",
        "fecha_corte",
        "horizonte",
        "fecha_target",
        "perfil",
        "regimen_actual",
        "modelo_ganador",
        "alpha_ml",
        "real_kwh",
        "pred_ml_kwh",
        "baseline_kwh",
        "pred_final_kwh",
        "error_kwh",
        "error_abs_kwh",
        "error_pct",
    ]
].copy()

real_vs_pred.to_parquet(
    RUTA_REAL_VS_PRED,
    index=False,
    engine="pyarrow",
)

display(
    real_vs_pred.head(20)
)

In [ ]:
# ============================================================
# 35. GRÁFICA: WAPE SISTEMA GANADOR VS BASELINE
# ============================================================

graf = (
    metricas_globales_backtest
    .set_index(
        "horizonte"
    )[
        [
            "WAPE_final_pct",
            "WAPE_ML_ganadores_pct",
            "WAPE_baseline_pct",
        ]
    ]
)

ax = graf.plot(
    marker="o",
    figsize=(11, 5),
)

ax.set_title(
    "Backtest: sistema ganador vs ML puro vs baseline"
)

ax.set_xlabel(
    "Horizonte (meses)"
)

ax.set_ylabel(
    "WAPE (%)"
)

ax.grid(
    alpha=0.25
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 36. GRÁFICA: REAL VS SISTEMA GANADOR EN BACKTEST
# ============================================================

resumen_backtest = (
    backtest
    .groupby(
        [
            "horizonte",
            "fecha_target",
        ],
        as_index=False,
    )
    .agg(
        real_total_kwh=(
            "real_kwh",
            "sum"
        ),
        pred_ml_total_kwh=(
            "pred_ml_kwh",
            "sum"
        ),
        pred_final_total_kwh=(
            "pred_final_kwh",
            "sum"
        ),
        baseline_total_kwh=(
            "baseline_kwh",
            "sum"
        ),
    )
    .sort_values(
        "fecha_target"
    )
)

plt.figure(
    figsize=(12, 5)
)

plt.plot(
    resumen_backtest[
        "fecha_target"
    ],
    resumen_backtest[
        "real_total_kwh"
    ],
    marker="o",
    linewidth=2,
    label="Real",
)

plt.plot(
    resumen_backtest[
        "fecha_target"
    ],
    resumen_backtest[
        "pred_final_total_kwh"
    ],
    marker="o",
    label="Sistema ganador",
)

plt.plot(
    resumen_backtest[
        "fecha_target"
    ],
    resumen_backtest[
        "pred_ml_total_kwh"
    ],
    marker="o",
    label="ML ganador sin blend",
)

plt.plot(
    resumen_backtest[
        "fecha_target"
    ],
    resumen_backtest[
        "baseline_total_kwh"
    ],
    marker="o",
    linestyle="--",
    label="Baseline",
)

plt.title(
    "Backtest: consumo real vs pronosticado"
)

plt.xlabel(
    "Mes objetivo"
)

plt.ylabel(
    "Consumo total (kWh)"
)

plt.legend()
plt.xticks(
    rotation=45
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 37. GRÁFICA: P3 GRANDES CONSUMIDORES EN BACKTEST
# ============================================================

p3_backtest = (
    backtest[
        backtest[
            "perfil"
        ].eq(
            "P3_GRANDE"
        )
    ]
    .groupby(
        [
            "horizonte",
            "fecha_target",
        ],
        as_index=False,
    )
    .agg(
        real_total_kwh=(
            "real_kwh",
            "sum"
        ),
        pred_ml_total_kwh=(
            "pred_ml_kwh",
            "sum"
        ),
        pred_final_total_kwh=(
            "pred_final_kwh",
            "sum"
        ),
        baseline_total_kwh=(
            "baseline_kwh",
            "sum"
        ),
    )
)

if not p3_backtest.empty:

    plt.figure(
        figsize=(12, 5)
    )

    plt.plot(
        p3_backtest[
            "fecha_target"
        ],
        p3_backtest[
            "real_total_kwh"
        ],
        marker="o",
        linewidth=2,
        label="Real P3",
    )

    plt.plot(
        p3_backtest[
            "fecha_target"
        ],
        p3_backtest[
            "pred_final_total_kwh"
        ],
        marker="o",
        label="Sistema ganador P3",
    )

    plt.plot(
        p3_backtest[
            "fecha_target"
        ],
        p3_backtest[
            "pred_ml_total_kwh"
        ],
        marker="o",
        label="ML ganador P3",
    )

    plt.plot(
        p3_backtest[
            "fecha_target"
        ],
        p3_backtest[
            "baseline_total_kwh"
        ],
        marker="o",
        linestyle="--",
        label="Baseline P3",
    )

    plt.title(
        "Backtest P3: grandes consumidores real vs pronosticado"
    )

    plt.xlabel(
        "Mes objetivo"
    )

    plt.ylabel(
        "Consumo total P3 (kWh)"
    )

    plt.legend()
    plt.xticks(
        rotation=45
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 38. GRÁFICA: REAL VS PRONOSTICADO CLIENTE A CLIENTE
# ============================================================

muestra_scatter = (
    real_vs_pred[
        [
            "real_kwh",
            "pred_final_kwh",
        ]
    ]
    .dropna()
    .sample(
        n=min(
            20_000,
            len(
                real_vs_pred
            ),
        ),
        random_state=SEED,
    )
)

limite = max(
    float(
        muestra_scatter[
            "real_kwh"
        ]
        .quantile(
            0.995
        )
    ),
    float(
        muestra_scatter[
            "pred_final_kwh"
        ]
        .quantile(
            0.995
        )
    ),
)

plt.figure(
    figsize=(7, 7)
)

plt.scatter(
    muestra_scatter[
        "real_kwh"
    ],
    muestra_scatter[
        "pred_final_kwh"
    ],
    alpha=0.20,
    s=8,
)

plt.plot(
    [
        0,
        limite,
    ],
    [
        0,
        limite,
    ],
    linestyle="--",
    label="Predicción perfecta",
)

plt.xlim(
    0,
    limite,
)

plt.ylim(
    0,
    limite,
)

plt.title(
    "Backtest: consumo real vs pronosticado por cliente"
)

plt.xlabel(
    "Real (kWh)"
)

plt.ylabel(
    "Pronosticado (kWh)"
)

plt.legend()

plt.tight_layout()
plt.show()

## Gráficas adicionales — comparación de errores, segmentos y selección (backtest)

Las siguientes gráficas usan exclusivamente el backtest (agosto 2025 a enero 2026), el
período que nunca se usó para elegir algoritmo ni `alpha`. Muestran, por segmento, cómo
se comportó el sistema ganador frente al baseline, qué algoritmo quedó asignado a cada
combinación `perfil × horizonte`, las series de real vs. pronosticado, y el desglose por
régimen de lectura (observado / reconstruido / sin dato).


In [ ]:
# ============================================================
# 33b. GRAFICA: WAPE FINAL VS BASELINE POR PERFIL (BACKTEST)
# ============================================================

perfiles_backtest_plot = [
    p for p in PERFILES
    if p in metricas_perfil_backtest["perfil"].unique()
]

n_perfiles = len(perfiles_backtest_plot)
n_filas = int(np.ceil(n_perfiles / 2))

fig, axes = plt.subplots(n_filas, 2, figsize=(14, 4.5 * n_filas), sharey=False)
axes = np.atleast_1d(axes).flatten()

ancho = 0.35

for ax, perfil in zip(axes, perfiles_backtest_plot):

    temp = (
        metricas_perfil_backtest[
            metricas_perfil_backtest["perfil"].eq(perfil)
        ]
        .sort_values("horizonte")
    )

    x = np.arange(len(temp))

    ax.bar(
        x - ancho / 2, temp["WAPE_final_pct"],
        width=ancho, label="Sistema (ganador)", color="#1565C0",
    )
    ax.bar(
        x + ancho / 2, temp["WAPE_baseline_pct"],
        width=ancho, label="Baseline", color="#9E9E9E",
    )

    for xi, (modelo, wape) in enumerate(
        zip(temp["modelo_ganador"], temp["WAPE_final_pct"])
    ):
        ax.text(
            xi - ancho / 2, wape + 0.5, modelo,
            ha="center", va="bottom", fontsize=8, rotation=90,
        )

    ax.set_xticks(x)
    ax.set_xticklabels(temp["horizonte"])
    ax.set_title(perfil)
    ax.set_xlabel("Horizonte (meses)")
    ax.set_ylabel("WAPE backtest (%)")
    ax.grid(alpha=0.25, axis="y")
    ax.legend(fontsize=8)

for ax in axes[n_perfiles:]:
    ax.axis("off")

fig.suptitle(
    "Backtest: WAPE del sistema ganador vs baseline, por perfil y horizonte",
    fontsize=13,
)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 33c. MAPA DE SELECCION: QUE ALGORITMO GANO (BACKTEST)
# ============================================================

pivot_wape_bt = (
    metricas_perfil_backtest
    .pivot(index="perfil", columns="horizonte", values="WAPE_final_pct")
    .reindex(PERFILES)
)

pivot_modelo_bt = (
    metricas_perfil_backtest
    .pivot(index="perfil", columns="horizonte", values="modelo_ganador")
    .reindex(PERFILES)
)

fig, ax = plt.subplots(figsize=(10, 5.5))

im = ax.imshow(
    pivot_wape_bt.to_numpy(dtype="float64"),
    cmap="RdYlGn_r",
    aspect="auto",
)

ax.set_xticks(range(len(pivot_wape_bt.columns)))
ax.set_xticklabels(pivot_wape_bt.columns)
ax.set_yticks(range(len(pivot_wape_bt.index)))
ax.set_yticklabels(pivot_wape_bt.index)
ax.set_xlabel("Horizonte (meses)")
ax.set_title("Backtest: algoritmo ganador y WAPE final del sistema (%)")

for i in range(pivot_wape_bt.shape[0]):
    for j in range(pivot_wape_bt.shape[1]):
        wape = pivot_wape_bt.iat[i, j]
        modelo = pivot_modelo_bt.iat[i, j]
        if pd.notna(wape):
            ax.text(
                j, i,
                f"{modelo}\n{wape:.1f}%",
                ha="center", va="center",
                fontsize=8, color="black",
            )

fig.colorbar(im, ax=ax, label="WAPE (%)")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 33d. GRAFICA: REAL VS PRONOSTICADO POR MODELO (BACKTEST)
# ============================================================

totales_modelo_backtest = (
    backtest
    .groupby(["modelo_ganador", "fecha_target"], as_index=False)
    .agg(
        real_total_kwh=("real_kwh", "sum"),
        pred_total_kwh=("pred_final_kwh", "sum"),
        baseline_total_kwh=("baseline_kwh", "sum"),
        n=("real_kwh", "size"),
    )
)

fig, ax = plt.subplots(figsize=(13, 6))

for modelo in totales_modelo_backtest["modelo_ganador"].unique():

    temp = (
        totales_modelo_backtest[
            totales_modelo_backtest["modelo_ganador"].eq(modelo)
        ]
        .sort_values("fecha_target")
    )

    ax.plot(
        temp["fecha_target"], temp["real_total_kwh"],
        marker="o", linewidth=2, label=f"Real ({modelo})",
    )
    ax.plot(
        temp["fecha_target"], temp["pred_total_kwh"],
        marker="s", linestyle="--", label=f"Pronóstico ({modelo})",
    )

ax.set_title(
    "Backtest: consumo real vs. pronosticado, agrupado por el algoritmo "
    "asignado a cada segmento"
)
ax.set_xlabel("Mes objetivo")
ax.set_ylabel("Consumo total (kWh)")
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.25)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 33e. GRAFICA: REAL VS PRONOSTICADO POR PERFIL (BACKTEST)
# ============================================================

for perfil in PERFILES:

    temp_perfil = (
        backtest[backtest["perfil"].eq(perfil)]
        .groupby("fecha_target", as_index=False)
        .agg(
            real_total_kwh=("real_kwh", "sum"),
            pred_total_kwh=("pred_final_kwh", "sum"),
            baseline_total_kwh=("baseline_kwh", "sum"),
        )
        .sort_values("fecha_target")
    )

    if temp_perfil.empty:
        continue

    plt.figure(figsize=(11, 5))

    plt.plot(
        temp_perfil["fecha_target"], temp_perfil["real_total_kwh"],
        marker="o", linewidth=2, label="Real",
    )
    plt.plot(
        temp_perfil["fecha_target"], temp_perfil["pred_total_kwh"],
        marker="o", label="Sistema (ganador)",
    )
    plt.plot(
        temp_perfil["fecha_target"], temp_perfil["baseline_total_kwh"],
        marker="o", linestyle="--", label="Baseline",
    )

    plt.title(f"Backtest — {perfil}: consumo real vs. pronosticado")
    plt.xlabel("Mes objetivo")
    plt.ylabel(f"Consumo total {perfil} (kWh)")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# 33f. GRAFICA: WAPE POR REGIMEN DE LECTURA (BACKTEST)
# ============================================================

tabla_regimen = (
    metricas_regimen
    .pivot(index="horizonte", columns="regimen", values="WAPE_pct")
)

colores_regimen = {
    "observado": "#2E7D32",
    "reconstruido": "#C62828",
    "sin_dato": "#616161",
}

ax = tabla_regimen.plot(
    kind="bar",
    figsize=(11, 5),
    color=[colores_regimen.get(c, "#455A64") for c in tabla_regimen.columns],
)

ax.set_yscale("log")
ax.set_title(
    "Backtest: WAPE por régimen de lectura (observado / reconstruido / sin dato)"
)
ax.set_xlabel("Horizonte (meses)")
ax.set_ylabel("WAPE (%) — escala log")
ax.legend(title="Régimen")
ax.grid(alpha=0.25, axis="y")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(
    "Nota: 'reconstruido' son lecturas imputadas, no observadas directamente. "
    "Un WAPE mucho más alto en ese grupo -sobre todo en horizontes largos- "
    "indica que la confiabilidad del pronóstico depende de si el histórico "
    "del cliente es lectura real o estimada."
)


# Reentrenamiento final y pronóstico real

Una vez fijados los ganadores y los `alpha`, se entrenan nuevamente **solo los modelos seleccionados** con todos los targets conocidos hasta el último mes del archivo.

Con el histórico actual, el corte esperado es enero de 2026 y se generan predicciones de febrero a julio de 2026.

Ni el algoritmo ganador ni el `alpha` se recalculan usando el backtest.

In [ ]:
# ============================================================
# 39. ENTRENAR MODELOS GANADORES FINALES
# ============================================================

FECHA_CORTE_FINAL = periodo_mes(
    periodo_max
)

modelos_finales = {}

print(
    "Fecha de corte final:",
    FECHA_CORTE_FINAL.strftime(
        "%Y-%m"
    )
)

for perfil in PERFILES_MODELADOS:

    for h in HORIZONTES:

        config = mapa_seleccion[
            (
                perfil,
                h,
            )
        ]

        algoritmo = config[
            "modelo"
        ]

        print("\n" + "=" * 75)

        print(
            f"FINAL | {perfil} | h={h} | {algoritmo}"
        )

        print("=" * 75)

        inicio = time.time()

        X_train, y_train, _ = (
            construir_train_perfil_h(
                perfil=perfil,
                horizonte=h,
                max_target_train=
                    FECHA_CORTE_FINAL,
                seed=(
                    SEED
                    + 500
                    + h
                ),
            )
        )

        bundle = entrenar_modelo_algoritmo(
            perfil=perfil,
            algoritmo=algoritmo,
            X=X_train,
            y=y_train,
        )

        modelos_finales[
            (
                perfil,
                h,
            )
        ] = bundle

        print(
            "Train:",
            X_train.shape,
            "| tiempo:",
            f"{(time.time() - inicio) / 60:.2f} min"
        )

        del X_train
        del y_train
        gc.collect()

In [ ]:
# ============================================================
# 40. GUARDAR BUNDLE DE MODELOS GANADORES
# ============================================================

joblib.dump(
    {
        "modelos":
            modelos_finales,

        "seleccion_modelos":
            seleccion_modelos,

        "mapa_seleccion":
            mapa_seleccion,

        "features":
            FEATURES,

        "config_perfiles":
            {
                "MIN_MESES_VALIDOS_12":
                    MIN_MESES_VALIDOS_12,

                "UMBRAL_MUY_BAJO_KWH":
                    UMBRAL_MUY_BAJO_KWH,

                "UMBRAL_ALTO_KWH":
                    UMBRAL_ALTO_KWH,

                "UMBRAL_GRANDE_KWH":
                    UMBRAL_GRANDE_KWH,

                "PCT_CEROS_INTERMITENTE":
                    PCT_CEROS_INTERMITENTE,
            },

        "fecha_corte":
            FECHA_CORTE_FINAL,

        "horizontes":
            HORIZONTES,

        "modelos_candidatos":
            MODELOS_CANDIDATOS,
    },
    RUTA_MODELOS,
)

print(
    "Bundle guardado:",
    RUTA_MODELOS
)

In [ ]:
# ============================================================
# 41. PRONÓSTICO FINAL t+1 A t+6
# ============================================================

perfiles_corte_final = calcular_perfiles(
    FECHA_CORTE_FINAL
)

partes_pred_final = []

for h in HORIZONTES:

    fecha_target = sumar_meses(
        FECHA_CORTE_FINAL,
        h,
    )

    for perfil in PERFILES:

        indices = (
            perfiles_corte_final.loc[
                perfiles_corte_final[
                    "perfil"
                ].eq(
                    perfil
                ),
                "indice",
            ]
            .to_numpy(
                dtype="int64"
            )
        )

        if len(indices) == 0:
            continue

        # Para pronóstico operativo conservamos clientes
        # presentes en el último mes.
        actual = valores_mes(
            FECHA_CORTE_FINAL,
            indices,
        )

        mask_activo = np.isfinite(
            actual
        )

        indices = indices[
            mask_activo
        ]

        actual = actual[
            mask_activo
        ]

        if len(indices) == 0:
            continue

        baseline = baseline_hibrido(
            FECHA_CORTE_FINAL,
            h,
            indices,
        )

        if perfil == "P4_INSUFICIENTE":

            algoritmo = "Baseline"

            alpha_ml = 0.0

            pred_ml = baseline.copy()

            pred_final = baseline.copy()

        else:

            config = mapa_seleccion[
                (
                    perfil,
                    h,
                )
            ]

            algoritmo = config[
                "modelo"
            ]

            alpha_ml = config[
                "alpha_ml"
            ]

            X, _ = crear_features(
                FECHA_CORTE_FINAL,
                h,
                indices,
            )

            pred_ml = predecir_bundle(
                modelos_finales[
                    (
                        perfil,
                        h,
                    )
                ],
                X,
            )

            pred_final = (
                alpha_ml
                * pred_ml
                + (
                    1
                    - alpha_ml
                )
                * baseline
            )

            pred_final = np.maximum(
                pred_final,
                0,
            ).astype(
                "float32"
            )

            del X

        regimen_actual = valores_regimen_mes(
            FECHA_CORTE_FINAL,
            indices,
        )

        regimen = np.where(
            regimen_actual == 1,
            "reconstruido",
            np.where(
                regimen_actual == 0,
                "observado",
                "sin_dato",
            ),
        )

        temp = pd.DataFrame(
            {
                "NIU":
                    nius[
                        indices
                    ],

                "fecha_corte":
                    FECHA_CORTE_FINAL,

                "horizonte":
                    h,

                "fecha_target":
                    fecha_target,

                "perfil":
                    perfil,

                "regimen_actual":
                    regimen,

                "modelo_seleccionado":
                    algoritmo,

                "alpha_ml":
                    alpha_ml,

                "consumo_actual_kwh":
                    actual,

                "pred_ml_kwh":
                    pred_ml,

                "baseline_kwh":
                    baseline,

                "pred_final_kwh":
                    pred_final,
            }
        )

        partes_pred_final.append(
            temp
        )

        del temp
        gc.collect()

pred_final = pd.concat(
    partes_pred_final,
    ignore_index=True,
)

display(
    pred_final.head(20)
)

print(
    "Filas pronóstico:",
    f"{len(pred_final):,}"
)

In [ ]:
# ============================================================
# 42. FORMATO ANCHO CON MODELO Y ALPHA POR HORIZONTE
# ============================================================

claves = [
    "NIU",
    "fecha_corte",
    "perfil",
    "regimen_actual",
]

pred_wide = (
    pred_final[
        claves
    ]
    .drop_duplicates()
    .copy()
)

for h in HORIZONTES:

    temp = (
        pred_final[
            pred_final[
                "horizonte"
            ].eq(
                h
            )
        ][
            claves
            + [
                "fecha_target",
                "modelo_seleccionado",
                "alpha_ml",
                "pred_final_kwh",
            ]
        ]
        .rename(
            columns={
                "fecha_target":
                    f"fecha_pred_{h}m",

                "modelo_seleccionado":
                    f"modelo_{h}m",

                "alpha_ml":
                    f"alpha_ml_{h}m",

                "pred_final_kwh":
                    f"pred_{h}m_kwh",
            }
        )
    )

    pred_wide = pred_wide.merge(
        temp,
        on=claves,
        how="left",
        validate="one_to_one",
    )

pred_wide[
    "promedio_pred_3m_kwh"
] = pred_wide[
    [
        "pred_1m_kwh",
        "pred_2m_kwh",
        "pred_3m_kwh",
    ]
].mean(
    axis=1
)

pred_wide[
    "promedio_pred_6m_kwh"
] = pred_wide[
    [
        "pred_1m_kwh",
        "pred_2m_kwh",
        "pred_3m_kwh",
        "pred_4m_kwh",
        "pred_5m_kwh",
        "pred_6m_kwh",
    ]
].mean(
    axis=1
)

display(
    pred_wide.head(20)
)

In [ ]:
# ============================================================
# 43. GUARDAR SALIDAS 3 Y 6 MESES
# ============================================================

cols_base = [
    "NIU",
    "fecha_corte",
    "perfil",
    "regimen_actual",
]

cols_3m = (
    cols_base
    + [
        "fecha_pred_1m",
        "modelo_1m",
        "alpha_ml_1m",
        "pred_1m_kwh",

        "fecha_pred_2m",
        "modelo_2m",
        "alpha_ml_2m",
        "pred_2m_kwh",

        "fecha_pred_3m",
        "modelo_3m",
        "alpha_ml_3m",
        "pred_3m_kwh",

        "promedio_pred_3m_kwh",
    ]
)

cols_6m = (
    cols_base
    + [
        "fecha_pred_1m",
        "modelo_1m",
        "alpha_ml_1m",
        "pred_1m_kwh",

        "fecha_pred_2m",
        "modelo_2m",
        "alpha_ml_2m",
        "pred_2m_kwh",

        "fecha_pred_3m",
        "modelo_3m",
        "alpha_ml_3m",
        "pred_3m_kwh",

        "fecha_pred_4m",
        "modelo_4m",
        "alpha_ml_4m",
        "pred_4m_kwh",

        "fecha_pred_5m",
        "modelo_5m",
        "alpha_ml_5m",
        "pred_5m_kwh",

        "fecha_pred_6m",
        "modelo_6m",
        "alpha_ml_6m",
        "pred_6m_kwh",

        "promedio_pred_3m_kwh",
        "promedio_pred_6m_kwh",
    ]
)

pred_3m = pred_wide[
    cols_3m
].copy()

pred_6m = pred_wide[
    cols_6m
].copy()

pred_3m.to_parquet(
    RUTA_PRED_3M,
    index=False,
    engine="pyarrow",
)

pred_6m.to_parquet(
    RUTA_PRED_6M,
    index=False,
    engine="pyarrow",
)

print("MODELADO SEGMENTADO COMPARATIVO TERMINADO")
print("=" * 75)

print("\nPerfiles y grandes consumidores:")
print(" •", RUTA_PERFILES_FINAL)
print(" •", RUTA_GRANDES_FINAL)
print(" •", RUTA_AUDITORIA_UMBRALES)

print("\nComparación de modelos:")
print(" •", RUTA_COMPARACION_VALIDACION)
print(" •", RUTA_SELECCION_MODELOS)
print(" •", RUTA_TOTALES_VALIDACION)

print("\nBacktest:")
print(" •", RUTA_METRICAS_BACKTEST)
print(" •", RUTA_METRICAS_PERFIL)
print(" •", RUTA_METRICAS_REGIMEN)
print(" •", RUTA_REAL_VS_PRED)

print("\nModelos:")
print(" •", RUTA_MODELOS)

print("\nPredicciones:")
print(" • 3 meses:", RUTA_PRED_3M)
print(" • 6 meses:", RUTA_PRED_6M)

# Cómo interpretar el resultado

El archivo más importante para comparar algoritmos es:

`comparacion_modelos_validacion.csv`

Contendrá una fila por:

`perfil × horizonte × modelo`

con:

- WAPE del ML puro;
- MAE;
- RMSE;
- R²;
- sesgo;
- WAPE del baseline;
- alpha óptimo;
- WAPE del blend;
- indicador de ganador.

El archivo:

`seleccion_modelo_por_perfil_horizonte.csv`

resume la decisión final.

Ejemplo conceptual:

| Perfil | Horizonte | Ganador | Alpha ML |
|---|---:|---|---:|
| P1_REGULAR | 1 | LightGBM | 0.90 |
| P1_REGULAR | 4 | CatBoost | 0.65 |
| P2_ALTO | 2 | XGBoost | 0.75 |
| P3_GRANDE | 6 | CatBoost | 0.25 |

Los valores reales serán calculados por el notebook.

## Regla metodológica

El algoritmo y el `alpha` se seleccionan únicamente con validación.

El backtest se utiliza para comprobar que esa decisión generaliza a meses posteriores y nunca para volver atrás y escoger otro ganador.

Esto mantiene una evaluación temporal limpia y evita fuga de información.

In [ ]:
# ============================================================
# DIAGNOSTICO: NATURALEZA DE LOS CONSUMOS EN 0 kWh
# ============================================================

es_valido = ~np.isnan(matriz_consumo)
es_cero = np.where(es_valido, matriz_consumo == 0, False)
es_positivo = np.where(es_valido, matriz_consumo > 0, False)

anterior_positivo = np.zeros_like(es_cero)
siguiente_positivo = np.zeros_like(es_cero)
anterior_positivo[:, 1:] = es_positivo[:, :-1]
siguiente_positivo[:, :-1] = es_positivo[:, 1:]

anterior_valido = np.zeros_like(es_cero)
siguiente_valido = np.zeros_like(es_cero)
anterior_valido[:, 1:] = es_valido[:, :-1]
siguiente_valido[:, :-1] = es_valido[:, 1:]

aislados = es_cero & anterior_positivo & siguiente_positivo
borde = es_cero & (~anterior_valido | ~siguiente_valido)
sostenidos = es_cero & ~aislados & ~borde

total_ceros = int(es_cero.sum())

print("DIAGNÓSTICO DE CONSUMOS EN 0 kWh")
print("-" * 60)
print(f"Total de lecturas válidas   : {int(es_valido.sum()):,}")
print(f"Total de lecturas en 0 kWh  : {total_ceros:,} "
      f"({total_ceros / es_valido.sum() * 100:.2f}% de las lecturas válidas)")
print()
print(f"Aislados (posible error de lectura)      : "
      f"{int(aislados.sum()):,} ({aislados.sum() / total_ceros * 100:.1f}%)")
print(f"Sostenidos (probable corte/desconexión)  : "
      f"{int(sostenidos.sum()):,} ({sostenidos.sum() / total_ceros * 100:.1f}%)")
print(f"En el borde de la serie (sin contexto)   : "
      f"{int(borde.sum()):,} ({borde.sum() / total_ceros * 100:.1f}%)")

# ------------------------------------------------------------
# Cruce contra el régimen observado / reconstruido, si existe
# ------------------------------------------------------------

if matriz_reconstruido is not None:

    regimen_en_ceros = matriz_reconstruido[es_cero]
    con_regimen = ~np.isnan(regimen_en_ceros)

    n_con_regimen = int(con_regimen.sum())
    n_observado = int((regimen_en_ceros[con_regimen] == 0).sum())
    n_reconstruido = int((regimen_en_ceros[con_regimen] == 1).sum())

    print()
    print("CRUCE CONTRA RÉGIMEN DE LECTURA (de los ceros con régimen conocido)")
    print("-" * 60)
    print(f"Con régimen conocido        : {n_con_regimen:,} de {total_ceros:,}")
    print(f"  Observados (lectura real) : {n_observado:,} "
          f"({n_observado / n_con_regimen * 100:.1f}%)")
    print(f"  Reconstruidos (imputados) : {n_reconstruido:,} "
          f"({n_reconstruido / n_con_regimen * 100:.1f}%)")

    # Aislados vs sostenidos, cruzados con régimen
    regimen_aislados = matriz_reconstruido[aislados]
    con_regimen_aislados = ~np.isnan(regimen_aislados)

    if con_regimen_aislados.any():
        pct_reconstruido_aislados = (
            (regimen_aislados[con_regimen_aislados] == 1).mean() * 100
        )
        print()
        print(f"De los ceros AISLADOS, % con régimen reconstruido: "
              f"{pct_reconstruido_aislados:.1f}%")
        print("(si este % es alto, refuerza la hipótesis de que los ceros "
              "aislados son artefactos de la reconstrucción trimestral, "
              "no consumo real)")
else:
    print()
    print("No hay columnas de procedencia (origen_consumo / consumo_imputado) "
          "en el archivo de entrada, así que no se puede cruzar contra régimen.")

# ------------------------------------------------------------
# Distribución de aislados por perfil de consumidor (contexto)
# ------------------------------------------------------------

niu_con_aislado = aislados.any(axis=1)
niu_con_sostenido = sostenidos.any(axis=1)

print()
print(f"NIU con al menos un cero aislado   : {int(niu_con_aislado.sum()):,}")
print(f"NIU con al menos una racha sostenida: {int(niu_con_sostenido.sum()):,}")

In [ ]:
# ============================================================
# DIAGNOSTICO: ¿LOS CLIENTES CON RACHA SOSTENIDA DE 0 VUELVEN A
# CONSUMIR, O DEJAN DE APARECER (POSIBLE BAJA DEL SERVICIO)?
# ============================================================

n_clientes, n_meses = es_cero.shape
indices_con_sostenida = np.flatnonzero(sostenidos.any(axis=1))

categoria = np.full(len(indices_con_sostenida), "", dtype=object)

for pos, i in enumerate(indices_con_sostenida):
    ultimo_idx_sost = int(np.max(np.flatnonzero(sostenidos[i])))

    if ultimo_idx_sost == n_meses - 1:
        categoria[pos] = "en_borde_ultimo_mes"
        continue

    resto_valido = es_valido[i, ultimo_idx_sost + 1:]
    resto_positivo = es_positivo[i, ultimo_idx_sost + 1:]

    if not resto_valido.any():
        categoria[pos] = "deja_de_aparecer_posible_baja"
    elif resto_positivo.any():
        categoria[pos] = "retoma_consumo"
    else:
        categoria[pos] = "sigue_en_cero_con_filas"

resumen_seguimiento = (
    pd.Series(categoria)
    .value_counts()
    .rename("n_clientes")
    .to_frame()
)
resumen_seguimiento["pct"] = (
    resumen_seguimiento["n_clientes"]
    / resumen_seguimiento["n_clientes"].sum()
    * 100
)

print(f"Clientes con al menos una racha sostenida: {len(indices_con_sostenida):,}")
display(resumen_seguimiento)

In [ ]:
# ============================================================
# CRUCE: "SIGUE_EN_CERO_CON_FILAS" vs ZONA RURAL Y PERFIL
# ============================================================

# 1. Aislar los NIU de cada categoría
niu_sigue_en_cero = nius[indices_con_sostenida[categoria == "sigue_en_cero_con_filas"]]
niu_retoma = nius[indices_con_sostenida[categoria == "retoma_consumo"]]

print(f"NIU 'sigue_en_cero_con_filas': {len(niu_sigue_en_cero):,}")
print(f"NIU 'retoma_consumo'        : {len(niu_retoma):,}")

# 2. Cruce contra zona rural
#    (requiere que 'serie' ya tenga la columna es_rural, es decir que hayas
#    regenerado el parquet con la celda 9b y recargado 'serie' en este kernel)
if "es_rural" in serie.columns:
    rural_por_niu = serie.drop_duplicates("NIU").set_index("NIU")["es_rural"]

    cruce_rural = (
        pd.Series(niu_sigue_en_cero)
        .map(rural_por_niu)
        .value_counts(dropna=False)
        .rename("n_clientes")
        .to_frame()
    )
    cruce_rural["pct"] = cruce_rural["n_clientes"] / cruce_rural["n_clientes"].sum() * 100

    print("\nDistribución es_rural entre 'sigue_en_cero_con_filas':")
    display(cruce_rural)
else:
    print("\n'serie' todavía no tiene la columna es_rural — regenera el parquet "
          "con la celda 9b en Preprocesamiento y vuelve a cargar 'serie' aquí.")

# 3. Cruce contra el perfil más reciente (tomado del backtest, horizonte=1)
perfil_por_niu = (
    backtest.loc[backtest["horizonte"] == 1]
    .drop_duplicates("NIU")
    .set_index("NIU")["perfil"]
)

cruce_perfil = (
    pd.Series(niu_sigue_en_cero)
    .map(perfil_por_niu)
    .value_counts(dropna=False)
    .rename("n_clientes")
    .to_frame()
)
cruce_perfil["pct"] = cruce_perfil["n_clientes"] / cruce_perfil["n_clientes"].sum() * 100

print("\nDistribución de perfil entre 'sigue_en_cero_con_filas':")
display(cruce_perfil)

# 4. Referencia: mismo cruce de perfil, pero para los que sí retoman consumo
perfil_retoma = (
    pd.Series(niu_retoma)
    .map(perfil_por_niu)
    .value_counts(dropna=False)
    .rename("n_clientes")
    .to_frame()
)
perfil_retoma["pct"] = perfil_retoma["n_clientes"] / perfil_retoma["n_clientes"].sum() * 100

print("\nDistribución de perfil entre 'retoma_consumo' (referencia):")
display(perfil_retoma)